In [1]:
# ============================================================
# 029_notion_clients_and_io.ipynb
# ============================================================
#
# Overview
# ----------------
# Shared Notion API I/O utilities for all researchOS pipeline notebooks.
# Provides a REST-based client, robust request/retry handling, error classification,
# schema introspection/validation, and CRUD wrappers for:
#   - Papers
#   - Events
#   - Monitoring Targets
#   - Monitoring Queue
#
# This notebook is designed to be SDK-independent and resilient for Daily runs.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Environment variables from env.txt:
#       NOTION_TOKEN, NOTION_VERSION,
#       NOTION_LIT_DB_ID, NOTION_EVENTS_DB_ID,
#       NOTION_MONITORING_TARGETS_DB_ID, NOTION_MONITORING_QUEUE_DB_ID
#   - Expected required properties per DB (researchOS schema + Plan A observability fields)
#
# Outputs:
#   - Reusable Notion HTTP client helpers (GET/POST/PATCH + retry + rate-limit)
#   - Error classification utilities (RETRYABLE / MANUAL_FIX / FATAL)
#   - Schema introspection + validation utilities (data_sources-aware)
#   - CRUD wrappers for each DB (create/query/update)
#   - Deduplication helpers (Dedup Key + URL/title normalization)
#   - Run ID logging utilities (stable ISO8601 UTC with 'Z')
#
# Structure
# ----------------
# Cell 01: Import dependencies and load environment variables (env.txt)
# Cell 02: Define required properties (researchOS schema + Plan A additions)
# Cell 03: REST-based auth + database access validation (SDK-independent)
# Cell 04: Core HTTP request wrapper (rate limit + retries + backoff)
# Cell 05: Error classification system (actionable categories)
# Cell 06: Schema validation utilities (compare expected vs actual)
# Cell 07: Property introspection utilities (dynamic type detection)
# Cell 08: Papers DB wrappers (INBOX ingest + updates)
# Cell 09: Events DB wrappers (daily monitoring events)
# Cell 10: Monitoring Targets DB wrappers
# Cell 11: Monitoring Queue DB wrappers (idempotent enqueue + updates)
# Cell 12: Deduplication helpers (Dedup Key + normalization)
# Cell 13: Logging utilities (RunContext + run_id correlation)
# Cell 14: Diagnostic test (auth/access/schema/introspection; data_sources-aware)
# Cell 15: Usage examples (schema-aligned)
#
# Notes
# ----------------
# - Authentication validation uses REST calls (/users/me), not a Notion SDK.
# - IMPORTANT: In this workspace, database property schema may not appear under
#   GET /databases/{id}. Instead, retrieve schema via the database's data source:
#     1) GET /databases/{id} -> data_sources[0].id
#     2) GET /data_sources/{data_source_id} -> properties
#   All introspection/validation should be data_sources-aware.
# - Rate limiting: enforce ~3 req/sec; retries with exponential backoff for 429/5xx.
# - Error categories:
#     RETRYABLE  : 429, 5xx, timeouts/connection errors
#     MANUAL_FIX : 400 validation, schema/property mismatch
#     FATAL      : 401/403 auth, 404 missing objects
# - Each DB wrapper should validate required properties before write operations.
# - Run ID is used for correlation across a single pipeline execution:
#   format = ISO8601 UTC with 'Z' suffix (e.g., 2026-01-25T05:59:07.123456Z).
# - Database ID constraint: NOTION_*_DB_ID must be database IDs (not page IDs).
# - Dedup strategy: write deterministic Dedup Key to enable idempotent Daily runs.


In [2]:
# ============================================================
# Cell 01 — Import dependencies and load environment variables
# ============================================================
# Overview:
#   Load environment variables from env.txt and import all standard libraries
#   needed for Notion API interaction, HTTP requests, error handling, and logging.
#
# Inputs / Outputs:
#   Inputs: env.txt file with NOTION_TOKEN, database IDs, NOTION_VERSION
#   Outputs: Configured environment variables, imported modules
#
# Notes:
#   - Uses requests library for REST API calls (SDK-independent)
#   - Time and datetime for retry logic and run ID generation
#   - JSON for payload serialization and response parsing
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import json
import time
import logging
from datetime import datetime, timezone
from typing import Dict, List, Optional, Any, Tuple
from enum import Enum

# --- HTTP and API interaction ---
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Load Notion credentials and configuration ---
NOTION_TOKEN = os.getenv('NOTION_TOKEN')
NOTION_VERSION = os.getenv('NOTION_VERSION', '2022-06-28')

# Database IDs
PAPERS_DB_ID = os.getenv('NOTION_LIT_DB_ID')
EVENTS_DB_ID = os.getenv('NOTION_EVENTS_DB_ID')
MONITORING_TARGETS_DB_ID = os.getenv('NOTION_MONITORING_TARGETS_DB_ID')
MONITORING_QUEUE_DB_ID = os.getenv('NOTION_MONITORING_QUEUE_DB_ID')

# --- Validate critical environment variables ---
if not NOTION_TOKEN:
    raise ValueError("NOTION_TOKEN not found in environment variables")

if not all([PAPERS_DB_ID, EVENTS_DB_ID, MONITORING_TARGETS_DB_ID, MONITORING_QUEUE_DB_ID]):
    missing = []
    if not PAPERS_DB_ID: missing.append('NOTION_PAPERS_DB_ID')
    if not EVENTS_DB_ID: missing.append('NOTION_EVENTS_DB_ID')
    if not MONITORING_TARGETS_DB_ID: missing.append('NOTION_MONITORING_TARGETS_DB_ID')
    if not MONITORING_QUEUE_DB_ID: missing.append('NOTION_MONITORING_QUEUE_DB_ID')
    raise ValueError(f"Missing database IDs in environment: {', '.join(missing)}")

# --- Configure logging ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

logger.info("Environment variables loaded successfully")
logger.info(f"Notion API Version: {NOTION_VERSION}")
logger.info(f"Database IDs configured: Papers, Events, Monitoring Targets, Monitoring Queue")


2026-01-27 15:50:14 [INFO] Environment variables loaded successfully
2026-01-27 15:50:14 [INFO] Notion API Version: 2025-09-03
2026-01-27 15:50:14 [INFO] Database IDs configured: Papers, Events, Monitoring Targets, Monitoring Queue


In [3]:
# ============================================================
# Cell 02 — Define database schemas and required properties
# ============================================================
# Overview:
#   Define the expected schema for each Notion database including
#   required properties, their types, and validation rules.
#   These schemas are used for validation before CRUD operations.
#
# Inputs / Outputs:
#   Inputs: None (hardcoded schemas based on researchOS design)
#   Outputs: Schema dictionaries for Papers, Events, Monitoring Targets, Monitoring Queue
#
# Notes:
#   - Property names must match exactly with Notion database configuration
#   - Types follow Notion API property type naming conventions
#   - Required properties are enforced during create/update operations
#   - Optional properties can be added later via introspection
#

# --- Papers Database Schema ---
# Current required = LIT_SCHEMA values + (Plan A additions)
PAPERS_SCHEMA = {
    "database_id": PAPERS_DB_ID,
    "required_properties": {
        # --- Existing (from LIT_SCHEMA) ---
        "Name": "title",
        "Created time": "created_time",
        "Authors & Year": "rich_text",
        "Tags": "multi_select",
        "PDF Link": "url",
        "Findings": "rich_text",
        "Core Idea": "rich_text",
        "Notes": "rich_text",
        "Methods": "rich_text",
        "Type": "select",
        "Source": "select",
        "Datasets": "rich_text",
        "Papers": "relation",

        # --- Plan A additions ---
        "Status": "select",
        "Dedup Key": "rich_text",
        "Source UID": "rich_text",
        "Ingested At": "date",
        "Run ID": "rich_text",
        "PDF Status": "select",
        "Slide 1 URL": "url",
    },
    "optional_properties": {
        # keep empty or add later; introspection can surface more
    },
}

# --- Monitoring Targets Database Schema ---
# Current required = REQUIRED_PROPS_MONITORING_TARGETS + (Plan A additions)
MONITORING_TARGETS_SCHEMA = {
    "database_id": MONITORING_TARGETS_DB_ID,
    "required_properties": {
        # --- Existing (from your REQUIRED_PROPS_MONITORING_TARGETS) ---
        "Name": "title",
        "Type": "select",
        "Status": "select",
        "Priority": "select",      # if you use number instead, change to "number"
        "Search Keywords": "rich_text",
        "Source URLs": "rich_text", # if you use URL property, change to "url"
        "Cadence": "select",
        "Last Checked": "date",
        "Next Check": "date",

        # --- Plan A additions ---
        "Enabled": "checkbox",
        "Source Type": "select",
        "Last Error": "rich_text",
        "Error Count": "number",
    },
    "optional_properties": {},
}

# --- Events Database Schema ---
# Current required = REQUIRED_PROPS_EVENTS + (Plan A additions)
EVENTS_SCHEMA = {
    "database_id": EVENTS_DB_ID,
    "required_properties": {
        # --- Existing (from your REQUIRED_PROPS_EVENTS) ---
        "Name": "title",
        "Date": "date",
        "Detected At": "date",
        "Target": "relation",
        "Event Type": "select",
        "Source URL": "url",
        "Source": "select",
        "Summary": "rich_text",
        "Confidence": "number",
        "Dedup Key": "rich_text",
        "Status": "select",

        # --- Plan A additions ---
        "Run ID": "rich_text",
        "Ingested At": "date",
        "Action Needed": "checkbox",
        "Related Papers": "relation",
    },
    "optional_properties": {},
}

# --- Monitoring Queue Database Schema ---
# Current required = REQUIRED_PROPS_MONITORING_QUEUE + (Plan A additions)
MONITORING_QUEUE_SCHEMA = {
    "database_id": MONITORING_QUEUE_DB_ID,
    "required_properties": {
        # --- Existing (from your REQUIRED_PROPS_MONITORING_QUEUE) ---
        "Name": "title",
        "Queue Type": "select",
        "Target": "relation",
        "Scheduled At": "date",
        "Status": "select",
        "Attempts": "number",
        "Max Attempts": "number",
        "Last Error": "rich_text",
        "Run ID": "rich_text",

        # --- Plan A additions ---
        "Dedup Key": "rich_text",
        "Payload": "rich_text",
        "Last Attempt At": "date",
        "Result ID": "rich_text",
        "Result URL": "url",
    },
    "optional_properties": {},
}

# --- Consolidated schema registry ---
SCHEMA_REGISTRY = {
    "papers": PAPERS_SCHEMA,
    "events": EVENTS_SCHEMA,
    "monitoring_targets": MONITORING_TARGETS_SCHEMA,
    "monitoring_queue": MONITORING_QUEUE_SCHEMA,
}
# ============================================================
# 029 — Add: update_paper_links (schema-safe PATCH)
# ============================================================

from typing import Optional, Dict, Any

def update_paper_links(
    page_id: str,
    pdf_link: Optional[str] = None,
    slide1_url: Optional[str] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Update an existing paper page with Drive URLs.
    - Uses schema introspection + safe_property_value (same philosophy as create)
    """
    if not page_id:
        raise ValueError("page_id is required")

    types = introspect_database_properties(PAPERS_DB, use_cache=True)

    notion_props: Dict[str, Any] = {}

    # PDF Link
    if pdf_link and PAPER_PROPS.get("pdf_link") in types:
        pn = PAPER_PROPS["pdf_link"]
        notion_props[pn] = safe_property_value(types[pn], pdf_link, pn)

    # Slide1 URL
    if slide1_url and PAPER_PROPS.get("slide1_url") in types:
        pn = PAPER_PROPS["slide1_url"]
        notion_props[pn] = safe_property_value(types[pn], slide1_url, pn)

    # Extra (optional): もし Notion DB 側に JSON / rich_text 用のプロパティを置いてるならここで書けます
    if extra:
        for prop_name, v in extra.items():
            if prop_name not in types:
                logger.warning("Extra property not found in schema: %s (skipping)", prop_name)
                continue
            notion_props[prop_name] = safe_property_value(types[prop_name], v, prop_name)

    if not notion_props:
        logger.info("No updatable properties (pdf_link/slide1_url missing or not in schema).")
        return {"success": True, "page_id": page_id, "updated": []}

    payload = {"properties": notion_props}
    page = notion_client.request("PATCH", f"/pages/{page_id}", json=payload)

    return {"success": True, "page_id": page_id, "updated": list(notion_props.keys())}

logger.info("Database schemas defined successfully")
logger.info("Schemas registered for: %s", ", ".join(SCHEMA_REGISTRY.keys()))


2026-01-27 15:50:46 [INFO] Database schemas defined successfully
2026-01-27 15:50:46 [INFO] Schemas registered for: papers, events, monitoring_targets, monitoring_queue


In [4]:
# ============================================================
# Cell 03 — Initialize Notion client configuration (REST) + auth/db validation
# ============================================================
# Overview:
#   Initialize a single Notion REST client (session-based) and validate:
#   - Authentication (/users/me)
#   - Database accessibility (/databases/{id})
#   This notebook should use notion_client.request(...) everywhere.
#
# Inputs / Outputs:
#   Inputs: NOTION_TOKEN, NOTION_VERSION, DB IDs
#   Outputs: notion_client (single entrypoint), validated connectivity
#
# Notes:
#   - Keep one HTTP stack (requests.Session) to avoid subtle divergence.
#   - Notion 2025+ DB schema lives in data_sources; DB access validation still uses /databases/{id}.
#   - Do NOT import a module named "notion" to avoid name conflicts.
#

import time
import requests
from typing import Optional, Dict, Any, Tuple
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

NOTION_API_BASE_URL = "https://api.notion.com/v1"

# --- Request headers template ---
BASE_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

logger.info("Notion API base URL: %s", NOTION_API_BASE_URL)
logger.info("Notion API version: %s", NOTION_VERSION)


class NotionClient:
    """
    Notion REST client for researchOS (session-based).

    - Single request() entrypoint
    - Uses requests.Session for connection pooling
    - Retries transient errors (429 / 5xx) with backoff
    - Returns raw JSON (never flattens to results)
    """

    def __init__(
        self,
        token: str,
        notion_version: str,
        base_url: str = "https://api.notion.com/v1",
        timeout: int = 30,
        max_retries: int = 3,
        pool_connections: int = 10,
        pool_maxsize: int = 20,
    ):
        if not token:
            raise ValueError("NOTION_TOKEN is required")

        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Notion-Version": notion_version,
                "Content-Type": "application/json",
            }
        )

        retry_strategy = Retry(
            total=max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET", "POST", "PATCH", "PUT", "DELETE"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            max_retries=retry_strategy,
            pool_connections=pool_connections,
            pool_maxsize=pool_maxsize,
        )
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def request(
        self,
        method: str,
        path: str,
        json: Optional[Dict[str, Any]] = None,
        debug: bool = False,
    ) -> Dict[str, Any]:
        url = f"{self.base_url}{path}"
        resp = self.session.request(method=method, url=url, json=json, timeout=self.timeout)

        if debug:
            print("\n[HTTP DEBUG]")
            print("method:", method)
            print("url:", url)
            print("status:", resp.status_code)
            print("content-type:", resp.headers.get("content-type"))
            print("raw text (first 400 chars):", resp.text[:400])

        # Explicit errors (Retry adapter already retried)
        if resp.status_code >= 400:
            raise RuntimeError(f"Notion API error {resp.status_code}: {resp.text}")

        return resp.json()


# --- Instantiate single client (use this everywhere) ---
notion_client = NotionClient(
    token=NOTION_TOKEN,
    notion_version=NOTION_VERSION,
    base_url=NOTION_API_BASE_URL,
)

logger.info("NotionClient initialized (session-based)")


def validate_notion_auth() -> Tuple[bool, Optional[str]]:
    """Validate Notion token via /users/me."""
    try:
        me = notion_client.request("GET", "/users/me")
        user_type = me.get("type", "unknown")
        user_id = me.get("id", "unknown")
        logger.info("Authentication validated: %s (ID: %s)", user_type, user_id)
        return True, None
    except Exception as e:
        return False, str(e)


def validate_database_access(db_id: str, db_name: str) -> Tuple[bool, Optional[str]]:
    """Validate DB is accessible via /databases/{id}."""
    try:
        db = notion_client.request("GET", f"/databases/{db_id}")
        title = (db.get("title", [{}]) or [{}])[0].get("plain_text", "Untitled")
        logger.info("Database accessible: %s ('%s')", db_name, title)
        return True, None
    except Exception as e:
        return False, str(e)


# --- Execute authentication validation ---
auth_valid, auth_error = validate_notion_auth()
if not auth_valid:
    logger.error("Notion authentication failed: %s", auth_error)
    raise RuntimeError(f"Notion authentication validation failed: {auth_error}")

logger.info("Notion API authentication validated successfully")

# --- Validate all configured databases ---
db_validation_results = {
    "Papers": validate_database_access(PAPERS_DB_ID, "papers"),
    "Events": validate_database_access(EVENTS_DB_ID, "events"),
    "Monitoring Targets": validate_database_access(MONITORING_TARGETS_DB_ID, "monitoring_targets"),
    "Monitoring Queue": validate_database_access(MONITORING_QUEUE_DB_ID, "monitoring_queue"),
}

failed_dbs = [(name, err) for name, (ok, err) in db_validation_results.items() if not ok]
if failed_dbs:
    logger.warning("Some databases could not be validated: %d/%d", len(failed_dbs), len(db_validation_results))
    for db_name, err in failed_dbs:
        logger.error("  - %s: %s", db_name, err)
    raise RuntimeError("Database validation failed")

logger.info("All %d databases validated successfully", len(db_validation_results))
logger.info("Notion client configuration complete")


2026-01-27 15:50:52 [INFO] Notion API base URL: https://api.notion.com/v1
2026-01-27 15:50:52 [INFO] Notion API version: 2025-09-03
2026-01-27 15:50:52 [INFO] NotionClient initialized (session-based)
2026-01-27 15:50:53 [INFO] Authentication validated: bot (ID: 01c14ba4-e8b5-47cb-818c-c3df8b5b79d9)
2026-01-27 15:50:53 [INFO] Notion API authentication validated successfully
2026-01-27 15:50:53 [INFO] Database accessible: papers ('Literature Database')
2026-01-27 15:50:55 [INFO] Database accessible: events ('EVENTS_DB')
2026-01-27 15:50:56 [INFO] Database accessible: monitoring_targets ('MONITORING_TARGETS_DB')
2026-01-27 15:50:56 [INFO] Database accessible: monitoring_queue ('MONITORING_QUEUE_DB')
2026-01-27 15:50:56 [INFO] All 4 databases validated successfully
2026-01-27 15:50:56 [INFO] Notion client configuration complete


In [5]:
# ============================================================
# Cell 04 — NotionClient.request with rate limit + retry + error classification
# ============================================================
# Overview:
#   Provide a single robust HTTP entrypoint for all Notion API calls:
#   - Client-side rate limiting (~3 req/s)
#   - Retry for 429 and 5xx with exponential backoff
#   - Respect Retry-After header when present
#   - Clear error classification for downstream logic
#
# Notes:
#   - Notion 2025+ schema lives in data_sources; this request method is generic.
#   - Always returns raw JSON dict on success (never flattens to results).
#

import time
import requests
from enum import Enum
from typing import Optional, Dict, Any, Tuple
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


class ErrorCategory(Enum):
    RETRYABLE = "retryable"      # 429, 5xx, transient network
    MANUAL_FIX = "manual_fix"    # 400 validation/schema/property mismatch
    FATAL = "fatal"              # 401/403 auth, 404 not found
    SUCCESS = "success"


class NotionAPIError(RuntimeError):
    def __init__(self, message: str, category: ErrorCategory, status_code: Optional[int] = None, payload: Optional[dict] = None):
        super().__init__(message)
        self.category = category
        self.status_code = status_code
        self.payload = payload or {}


class NotionClient:
    def __init__(
        self,
        token: str,
        notion_version: str,
        base_url: str = "https://api.notion.com/v1",
        timeout: int = 30,
        max_retries: int = 3,
        min_request_interval: float = 0.34,  # ~3 req/sec
        pool_connections: int = 10,
        pool_maxsize: int = 20,
    ):
        if not token:
            raise ValueError("NOTION_TOKEN is required")

        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries
        self.min_request_interval = min_request_interval
        self._last_request_time = 0.0

        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Notion-Version": notion_version,
                "Content-Type": "application/json",
            }
        )

        # Basic adapter-level retry for connection-level issues.
        # We still do application-level retry below (429/5xx/Retry-After).
        adapter_retry = Retry(
            total=0,  # keep 0 to avoid double-retry confusion; we handle retries ourselves
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            max_retries=adapter_retry,
            pool_connections=pool_connections,
            pool_maxsize=pool_maxsize,
        )
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def _enforce_rate_limit(self):
        now = time.time()
        dt = now - self._last_request_time
        if dt < self.min_request_interval:
            time.sleep(self.min_request_interval - dt)
        self._last_request_time = time.time()

    def _classify_error(self, status_code: int, error_json: Optional[dict]) -> Tuple[ErrorCategory, str]:
        # Notion error shape usually includes: {"object":"error","status":...,"code":...,"message":...}
        msg = ""
        if isinstance(error_json, dict):
            msg = error_json.get("message") or ""
            code = error_json.get("code") or "unknown"
        else:
            code = "unknown"

        if status_code in (401, 403):
            return ErrorCategory.FATAL, f"Auth/permission error ({status_code}) code={code} msg={msg}"
        if status_code == 404:
            return ErrorCategory.FATAL, f"Not found ({status_code}) code={code} msg={msg}"
        if status_code == 400:
            return ErrorCategory.MANUAL_FIX, f"Validation/schema error ({status_code}) code={code} msg={msg}"
        if status_code == 429 or status_code >= 500:
            return ErrorCategory.RETRYABLE, f"Retryable error ({status_code}) code={code} msg={msg}"
        if 400 <= status_code < 500:
            return ErrorCategory.MANUAL_FIX, f"Client error ({status_code}) code={code} msg={msg}"
        return ErrorCategory.FATAL, f"Unexpected error ({status_code}) code={code} msg={msg}"

    def request(
        self,
        method: str,
        path: str,
        json: Optional[Dict[str, Any]] = None,
        debug: bool = False,
    ) -> Dict[str, Any]:
        method = method.upper()
        if method not in ("GET", "POST", "PATCH", "DELETE", "PUT"):
            raise ValueError(f"Invalid HTTP method: {method}")

        url = f"{self.base_url}{path}"
        backoff = 1.0

        for attempt in range(1, self.max_retries + 2):  # e.g., retries=3 => attempts up to 4
            try:
                self._enforce_rate_limit()

                resp = self.session.request(
                    method=method,
                    url=url,
                    json=json,
                    timeout=self.timeout,
                )

                if debug:
                    print("\n[HTTP DEBUG]")
                    print("method:", method, "url:", url, "status:", resp.status_code)
                    print("content-type:", resp.headers.get("content-type"))
                    print("raw text (first 400):", resp.text[:400])

                # Success
                if resp.status_code in (200, 201):
                    return resp.json()

                # Parse error body if possible
                err_json = None
                try:
                    err_json = resp.json()
                except Exception:
                    err_json = {"message": resp.text}

                # Rate limit
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = float(retry_after) if retry_after else backoff
                    if attempt <= self.max_retries + 1:
                        logger.warning("429 rate limited. wait=%.1fs attempt=%d url=%s", wait, attempt, path)
                        time.sleep(wait)
                        backoff *= 2
                        continue

                # 5xx retry
                if 500 <= resp.status_code < 600:
                    if attempt <= self.max_retries + 1:
                        logger.warning("5xx server error %d. backoff=%.1fs attempt=%d url=%s", resp.status_code, backoff, attempt, path)
                        time.sleep(backoff)
                        backoff *= 2
                        continue

                # Non-retryable
                category, msg = self._classify_error(resp.status_code, err_json)
                raise NotionAPIError(msg, category=category, status_code=resp.status_code, payload=err_json)

            except requests.exceptions.Timeout as e:
                if attempt <= self.max_retries + 1:
                    logger.warning("Timeout. backoff=%.1fs attempt=%d url=%s", backoff, attempt, path)
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                raise NotionAPIError(f"Timeout after retries: {e}", category=ErrorCategory.RETRYABLE)

            except requests.exceptions.ConnectionError as e:
                if attempt <= self.max_retries + 1:
                    logger.warning("Connection error. backoff=%.1fs attempt=%d url=%s", backoff, attempt, path)
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                raise NotionAPIError(f"Connection error after retries: {e}", category=ErrorCategory.RETRYABLE)

        raise NotionAPIError("Unexpected failure in NotionClient.request()", category=ErrorCategory.FATAL)


logger.info("NotionClient request wrapper ready (rate limit + retry + classification)")


2026-01-27 15:51:24 [INFO] NotionClient request wrapper ready (rate limit + retry + classification)


In [6]:
# ============================================================
# Cell 05 — Implement error classification system
# ============================================================
# Overview:
#   Classify Notion API errors into categories (RETRYABLE, MANUAL_FIX, FATAL)
#   to guide automatic retry logic and manual intervention requirements.
#
# Notes:
#   - Works with:
#       (a) requests.Response (if you ever handle raw responses)
#       (b) Exceptions from NotionClient.request (recommended)
#   - Notion 2025+ schema lives in data_sources; classification is transport-agnostic.
#

from dataclasses import dataclass
from typing import Optional, Dict, Any, List
import requests

# ErrorCategory should already exist (Cell04). If not, fail loudly.
try:
    ErrorCategory
except NameError as e:
    raise RuntimeError("ErrorCategory must be defined before Cell 05") from e


@dataclass
class NotionErrorInfo:
    category: ErrorCategory
    status_code: Optional[int] = None
    error_code: str = "unknown"
    message: str = ""
    retry_after: Optional[float] = None
    raw_response: str = ""

    def is_retryable(self) -> bool:
        return self.category == ErrorCategory.RETRYABLE

    def requires_manual_fix(self) -> bool:
        return self.category == ErrorCategory.MANUAL_FIX

    def is_fatal(self) -> bool:
        return self.category == ErrorCategory.FATAL


def _parse_notion_error_payload(text: str) -> Dict[str, Any]:
    """
    Best-effort parse of Notion error JSON body.
    """
    try:
        return requests.models.complexjson.loads(text)  # avoids importing json explicitly
    except Exception:
        return {"message": text}


def classify_from_response(response: requests.Response) -> NotionErrorInfo:
    status_code = response.status_code
    retry_after = None

    # Parse JSON if possible
    error_data: Dict[str, Any] = {}
    if response.text:
        error_data = _parse_notion_error_payload(response.text)

    error_code = error_data.get("code", "unknown")
    error_message = error_data.get("message", response.text or "")

    if status_code == 429:
        ra = response.headers.get("Retry-After")
        try:
            retry_after = float(ra) if ra is not None else 60.0
        except Exception:
            retry_after = 60.0
        return NotionErrorInfo(
            category=ErrorCategory.RETRYABLE,
            status_code=status_code,
            error_code="rate_limited",
            message=f"Rate limited (429). Retry after {retry_after}s. {error_message}",
            retry_after=retry_after,
            raw_response=response.text or "",
        )

    if 500 <= status_code < 600:
        return NotionErrorInfo(
            category=ErrorCategory.RETRYABLE,
            status_code=status_code,
            error_code=error_code,
            message=f"Server error ({status_code}): {error_message}",
            raw_response=response.text or "",
        )

    if status_code in (401, 403):
        return NotionErrorInfo(
            category=ErrorCategory.FATAL,
            status_code=status_code,
            error_code=error_code,
            message=f"Auth/permission error ({status_code}): {error_message}",
            raw_response=response.text or "",
        )

    if status_code == 404:
        return NotionErrorInfo(
            category=ErrorCategory.FATAL,
            status_code=status_code,
            error_code=error_code,
            message=f"Not found (404): {error_message}",
            raw_response=response.text or "",
        )

    if status_code == 400:
        # Treat as manual fix (schema/property/payload issues)
        return NotionErrorInfo(
            category=ErrorCategory.MANUAL_FIX,
            status_code=status_code,
            error_code=error_code,
            message=f"Bad request / validation (400): {error_message}",
            raw_response=response.text or "",
        )

    if 400 <= status_code < 500:
        return NotionErrorInfo(
            category=ErrorCategory.MANUAL_FIX,
            status_code=status_code,
            error_code=error_code,
            message=f"Client error ({status_code}): {error_message}",
            raw_response=response.text or "",
        )

    return NotionErrorInfo(
        category=ErrorCategory.FATAL,
        status_code=status_code,
        error_code="unexpected_status",
        message=f"Unexpected status ({status_code}): {error_message}",
        raw_response=response.text or "",
    )


def classify_from_exception(exception: Exception) -> NotionErrorInfo:
    """
    Preferred path when using NotionClient.request() that raises exceptions.
    Supports:
    - requests Timeout / ConnectionError
    - NotionAPIError (if defined in Cell04)
    - generic Exception fallback
    """
    # Network exceptions
    if isinstance(exception, requests.exceptions.Timeout):
        return NotionErrorInfo(
            category=ErrorCategory.RETRYABLE,
            error_code="timeout",
            message=f"Request timeout: {exception}",
        )
    if isinstance(exception, requests.exceptions.ConnectionError):
        return NotionErrorInfo(
            category=ErrorCategory.RETRYABLE,
            error_code="connection_error",
            message=f"Connection error: {exception}",
        )

    # NotionClient error (if available)
    notion_api_error_cls = globals().get("NotionAPIError")
    if notion_api_error_cls is not None and isinstance(exception, notion_api_error_cls):
        # Expect attributes: category, status_code, payload
        category = getattr(exception, "category", ErrorCategory.FATAL)
        status_code = getattr(exception, "status_code", None)
        payload = getattr(exception, "payload", {}) or {}
        error_code = payload.get("code", "unknown") if isinstance(payload, dict) else "unknown"
        message = str(exception)
        return NotionErrorInfo(
            category=category,
            status_code=status_code,
            error_code=error_code,
            message=message,
            raw_response=str(payload)[:2000],
        )

    # Fallback
    return NotionErrorInfo(
        category=ErrorCategory.FATAL,
        error_code="unknown_exception",
        message=f"Unexpected exception: {exception}",
    )


def log_notion_error(error_info: NotionErrorInfo, context: str = "", level: str = "error") -> None:
    context_str = f" [{context}]" if context else ""
    msg = f"{error_info.category.value.upper()}{context_str}: {error_info.message}"

    if level == "debug":
        logger.debug(msg)
    elif level == "info":
        logger.info(msg)
    elif level == "warning":
        logger.warning(msg)
    else:
        logger.error(msg)

    # Extra debug detail
    if error_info.status_code is not None:
        logger.debug("  status_code=%s", error_info.status_code)
    if error_info.error_code and error_info.error_code != "unknown":
        logger.debug("  error_code=%s", error_info.error_code)
    if error_info.retry_after is not None:
        logger.debug("  retry_after=%ss", error_info.retry_after)


class ErrorStats:
    def __init__(self, max_recent: int = 50):
        self.counts = {
            ErrorCategory.RETRYABLE: 0,
            ErrorCategory.MANUAL_FIX: 0,
            ErrorCategory.FATAL: 0,
            ErrorCategory.SUCCESS: 0,
        }
        self.recent_errors: List[NotionErrorInfo] = []
        self.max_recent = max_recent

    def record(self, error_info: NotionErrorInfo) -> None:
        self.counts[error_info.category] += 1
        self.recent_errors.append(error_info)
        if len(self.recent_errors) > self.max_recent:
            self.recent_errors = self.recent_errors[-self.max_recent :]

    def get_summary(self) -> Dict[str, Any]:
        return {
            "total_errors": int(sum(self.counts.values())),
            "by_category": {cat.value: int(count) for cat, count in self.counts.items()},
            "recent_count": len(self.recent_errors),
        }

    def reset(self) -> None:
        for cat in self.counts:
            self.counts[cat] = 0
        self.recent_errors = []


error_stats = ErrorStats()

logger.info("Error classification system initialized (response + exception compatible)")


2026-01-27 15:51:37 [INFO] Error classification system initialized (response + exception compatible)


In [39]:
# ============================================================
# Cell 06 — Implement database schema validation utilities (Data Source aware)
# ============================================================
# Overview:
#   Validate Notion schemas against SCHEMA_REGISTRY by reading schema
#   from Notion Data Sources (Notion 2025+).
#
# Notes:
#   - DB retrieve (/databases/{id}) no longer includes "properties"
#   - Schema lives in data_sources:
#       /databases/{id} -> data_sources[0].id
#       /data_sources/{id} -> properties
#   - This cell standardizes:
#       - introspection (property name -> type)
#       - schema validation results (missing + mismatches)
#

from typing import Dict, Any, List, Tuple, Optional

# ---------- Data Source helpers ----------
_DATA_SOURCE_ID_CACHE: Dict[str, str] = {}

def get_primary_data_source_id(database_id: str, use_cache: bool = True) -> str:
    if use_cache and database_id in _DATA_SOURCE_ID_CACHE:
        return _DATA_SOURCE_ID_CACHE[database_id]

    db = notion_client.request("GET", f"/databases/{database_id}")
    data_sources = db.get("data_sources", []) or []
    if not data_sources:
        raise RuntimeError(f"No data_sources found for database {database_id}")

    ds_id = data_sources[0]["id"]
    _DATA_SOURCE_ID_CACHE[database_id] = ds_id
    return ds_id


def get_data_source_schema(database_id: str) -> Dict[str, Any]:
    """
    Retrieve the Data Source object that contains schema properties.
    """
    ds_id = get_primary_data_source_id(database_id)
    return notion_client.request("GET", f"/data_sources/{ds_id}")


def introspect_db_properties(database_id: str) -> Dict[str, str]:
    """
    Returns {property_name: property_type} from Data Source schema.
    """
    ds = get_data_source_schema(database_id)
    props = ds.get("properties", {}) or {}
    return {name: meta.get("type", "unknown") for name, meta in props.items()}


# ---------- Validation result ----------
class SchemaValidationResult:
    """
    Results from database schema validation.

    Attributes:
        is_valid: True if schema satisfies required properties
        missing: list of (prop_name, expected_type)
        mismatches: list of (prop_name, expected_type, actual_type)
    """
    def __init__(
        self,
        is_valid: bool,
        missing: List[Tuple[str, str]],
        mismatches: List[Tuple[str, str, str]],
    ):
        self.is_valid = is_valid
        self.missing = missing
        self.mismatches = mismatches

    def get_summary(self) -> Dict[str, Any]:
        return {
            "is_valid": self.is_valid,
            "missing_count": len(self.missing),
            "mismatch_count": len(self.mismatches),
        }

    def __repr__(self) -> str:
        status = "VALID" if self.is_valid else "INVALID"
        return f"SchemaValidationResult({status}, {len(self.missing)} missing, {len(self.mismatches)} mismatches)"


# ---------- Core schema validation ----------
def validate_database_schema(db_name: str, strict: bool = False) -> SchemaValidationResult:
    """
    Validate a single database schema by comparing SCHEMA_REGISTRY required_properties
    to actual properties introspected from the database's primary data source.

    strict=False:
        - Only checks existence (missing)
        - Does NOT fail on type mismatches
    strict=True:
        - Also checks type mismatches
    """
    schema = SCHEMA_REGISTRY[db_name]
    db_id = schema["database_id"]
    required = schema["required_properties"]  # {prop_name: expected_type}

    actual = introspect_db_properties(db_id)  # {prop_name: actual_type}

    missing: List[Tuple[str, str]] = []
    mismatches: List[Tuple[str, str, str]] = []

    for prop_name, expected_type in required.items():
        if prop_name not in actual:
            missing.append((prop_name, expected_type))
        else:
            actual_type = actual.get(prop_name, "unknown")
            if strict and expected_type and actual_type != expected_type:
                mismatches.append((prop_name, expected_type, actual_type))

    is_valid = (len(missing) == 0) and (len(mismatches) == 0 or not strict)

    # Log details (helpful but not too noisy)
    if missing:
        for p, t in missing[:20]:
            logger.warning("Missing required property: %s (%s)", p, t)
        if len(missing) > 20:
            logger.warning("... %d more missing properties", len(missing) - 20)

    if strict and mismatches:
        for p, exp, act in mismatches[:20]:
            logger.warning("Type mismatch: %s expected=%s actual=%s", p, exp, act)
        if len(mismatches) > 20:
            logger.warning("... %d more type mismatches", len(mismatches) - 20)

    return SchemaValidationResult(is_valid=is_valid, missing=missing, mismatches=mismatches)


def validate_all_databases(strict: bool = False) -> Dict[str, SchemaValidationResult]:
    """
    Validate all databases in SCHEMA_REGISTRY.
    Returns dict of {db_name: SchemaValidationResult}.
    """
    logger.info("Starting batch schema validation for all databases")
    results: Dict[str, SchemaValidationResult] = {}

    for db_name in SCHEMA_REGISTRY.keys():
        logger.info("Validating schema for: %s", db_name)
        results[db_name] = validate_database_schema(db_name, strict=strict)

        if not results[db_name].is_valid:
            logger.error(
                "Schema validation FAILED for %s (%s): %s",
                db_name,
                SCHEMA_REGISTRY[db_name]["database_id"],
                results[db_name],
            )

    valid_count = sum(1 for r in results.values() if r.is_valid)
    logger.info("Schema validation complete: %d/%d valid", valid_count, len(results))
    return results


def introspect_all_databases(use_cache: bool = True) -> Dict[str, Dict[str, str]]:
    """
    Introspect properties for all databases: {db_name: {prop_name: type}}.
    """
    out: Dict[str, Dict[str, str]] = {}
    for db_name, schema in SCHEMA_REGISTRY.items():
        db_id = schema["database_id"]
        prop_types = introspect_db_properties(db_id)
        out[db_name] = prop_types
        logger.info("Introspected %d properties from %s", len(prop_types), db_name)
    return out


logger.info("Database schema validation utilities initialized (Data Source aware)")
logger.info("Functions: validate_database_schema(), validate_all_databases(), introspect_all_databases()")


2026-01-26 12:42:57 [INFO] Database schema validation utilities initialized (Data Source aware)
2026-01-26 12:42:57 [INFO] Functions: validate_database_schema(), validate_all_databases(), introspect_all_databases()


In [7]:
# ============================================================
# Cell 07 — Property introspection for dynamic type detection (Data Source aware)
# ============================================================
# Overview:
#   Introspect Notion properties dynamically (name -> type + config),
#   cache results, and provide safe property value constructors for CRUD.
#
# Notes:
#   - Notion 2025+ schema is stored in Data Sources, not in /databases/{id}.
#   - Uses notion_client.request(...) as the single HTTP entrypoint.
#

from typing import Dict, Any, Optional, List

# ---------------------------------------------------------------------
# Cache (by database_id)
# ---------------------------------------------------------------------
_property_types_cache: Dict[str, Dict[str, str]] = {}
_property_schema_cache: Dict[str, Dict[str, Any]] = {}   # stores full data_source schema if needed


def clear_property_cache() -> None:
    _property_types_cache.clear()
    _property_schema_cache.clear()
    logger.debug("Property cache cleared")


def introspect_database_properties(database_id: str, use_cache: bool = True) -> Dict[str, str]:
    """
    Returns {property_name: property_type} using Data Source schema.
    """
    if use_cache and database_id in _property_types_cache:
        return _property_types_cache[database_id]

    # Data Source schema (Cell 06 helper)
    ds = get_data_source_schema(database_id)  # returns data source object containing 'properties'
    props = ds.get("properties", {}) or {}
    prop_types = {name: meta.get("type", "unknown") for name, meta in props.items()}

    _property_types_cache[database_id] = prop_types
    _property_schema_cache[database_id] = ds

    logger.info("Introspected %d properties from database %s", len(prop_types), database_id)
    return prop_types


class PropertyMetadata:
    """
    Extended metadata for a Notion property.
    """
    def __init__(self, name: str, prop_type: str, config: Dict[str, Any], is_required: bool = False):
        self.name = name
        self.prop_type = prop_type
        self.config = config
        self.is_required = is_required
        self.options: List[str] = []

        if prop_type in ("select", "multi_select"):
            type_config = config.get(prop_type, {}) or {}
            self.options = [opt.get("name") for opt in type_config.get("options", []) if opt.get("name")]

    def __repr__(self) -> str:
        req = " [REQUIRED]" if self.is_required else ""
        return f"PropertyMetadata({self.name}: {self.prop_type}{req})"

    def get_summary(self) -> Dict[str, Any]:
        out = {"name": self.name, "type": self.prop_type, "required": self.is_required}
        if self.options:
            out["options"] = self.options
        return out


def get_property_metadata(
    database_id: str,
    expected_schema: Optional[Dict[str, Any]] = None,
    use_cache: bool = True,
) -> Dict[str, PropertyMetadata]:
    """
    Returns {property_name: PropertyMetadata} from Data Source schema.
    """
    if use_cache and database_id in _property_schema_cache:
        ds = _property_schema_cache[database_id]
    else:
        ds = get_data_source_schema(database_id)
        _property_schema_cache[database_id] = ds

    props = ds.get("properties", {}) or {}

    required_set = set()
    if expected_schema:
        required_set = set((expected_schema.get("required_properties", {}) or {}).keys())

    meta: Dict[str, PropertyMetadata] = {}
    for prop_name, prop_config in props.items():
        ptype = prop_config.get("type", "unknown")
        meta[prop_name] = PropertyMetadata(
            name=prop_name,
            prop_type=ptype,
            config=prop_config,
            is_required=(prop_name in required_set),
        )

    logger.debug("Retrieved metadata for %d properties from %s", len(meta), database_id)
    return meta


# ---------------------------------------------------------------------
# Property value constructors (safe)
# ---------------------------------------------------------------------
def safe_property_value(prop_type: str, value: Any, property_name: str = "unknown") -> Dict[str, Any]:
    """
    Convert a Python value into a Notion 'properties' payload snippet for the given prop type.
    """
    try:
        if prop_type == "title":
            if value is None:
                raise ValueError("Title cannot be None")
            if isinstance(value, str):
                return {"title": [{"text": {"content": value}}]}
            if isinstance(value, list):
                return {"title": value}
            raise ValueError(f"Title must be str or list, got {type(value)}")

        if prop_type == "rich_text":
            if value is None:
                return {"rich_text": []}
            if isinstance(value, str):
                return {"rich_text": [{"text": {"content": value}}]}
            if isinstance(value, list):
                return {"rich_text": value}
            raise ValueError(f"rich_text must be str or list, got {type(value)}")

        if prop_type == "select":
            if value is None:
                return {"select": None}
            if isinstance(value, str):
                return {"select": {"name": value}}
            raise ValueError(f"select must be str or None, got {type(value)}")

        if prop_type == "multi_select":
            if value is None:
                return {"multi_select": []}
            if isinstance(value, str):
                return {"multi_select": [{"name": value}]}
            if isinstance(value, list):
                return {"multi_select": [{"name": v} for v in value]}
            raise ValueError(f"multi_select must be list/str/None, got {type(value)}")

        if prop_type == "date":
            if value is None:
                return {"date": None}
            if isinstance(value, str):
                return {"date": {"start": value}}
            if isinstance(value, dict):
                return {"date": value}
            raise ValueError(f"date must be str/dict/None, got {type(value)}")

        if prop_type == "number":
            if value is None:
                return {"number": None}
            if isinstance(value, (int, float)):
                return {"number": value}
            raise ValueError(f"number must be int/float/None, got {type(value)}")

        if prop_type == "url":
            if value is None:
                return {"url": None}
            if isinstance(value, str):
                return {"url": value}
            raise ValueError(f"url must be str/None, got {type(value)}")

        if prop_type == "checkbox":
            if isinstance(value, bool):
                return {"checkbox": value}
            raise ValueError(f"checkbox must be bool, got {type(value)}")

        if prop_type == "relation":
            if value is None:
                return {"relation": []}
            if isinstance(value, str):
                return {"relation": [{"id": value}]}
            if isinstance(value, list):
                return {"relation": [{"id": v} for v in value]}
            raise ValueError(f"relation must be list/str/None, got {type(value)}")

        # Pass-through for unsupported types (keep it visible)
        logger.warning("Unsupported property type '%s' for %s; passing through", prop_type, property_name)
        return {prop_type: value}

    except Exception as e:
        raise ValueError(f"Failed to construct property value for '{property_name}' (type: {prop_type}): {e}") from e


def introspect_all_databases(use_cache: bool = True) -> Dict[str, Dict[str, str]]:
    """
    Returns: {db_name: {prop_name: type}}
    """
    out: Dict[str, Dict[str, str]] = {}
    for db_name, schema in SCHEMA_REGISTRY.items():
        db_id = schema["database_id"]
        out[db_name] = introspect_database_properties(db_id, use_cache=use_cache)
        logger.info("Introspected %d properties for %s", len(out[db_name]), db_name)
    return out


def print_property_report(metadata: Dict[str, PropertyMetadata]) -> None:
    print("\n" + "=" * 70)
    print("DATABASE PROPERTY METADATA")
    print("=" * 70)

    required = [m for m in metadata.values() if m.is_required]
    optional = [m for m in metadata.values() if not m.is_required]

    if required:
        print(f"\nRequired Properties ({len(required)}):")
        print("-" * 70)
        for prop in sorted(required, key=lambda p: p.name):
            opts = ""
            if prop.options:
                preview = ", ".join(prop.options[:3])
                opts = f" [Options: {preview}{'...' if len(prop.options) > 3 else ''}]"
            print(f"  {prop.name}: {prop.prop_type}{opts}")

    if optional:
        print(f"\nOptional Properties ({len(optional)}):")
        print("-" * 70)
        for prop in sorted(optional, key=lambda p: p.name):
            opts = ""
            if prop.options:
                preview = ", ".join(prop.options[:3])
                opts = f" [Options: {preview}{'...' if len(prop.options) > 3 else ''}]"
            print(f"  {prop.name}: {prop.prop_type}{opts}")

    print("\n" + "=" * 70 + "\n")


logger.info("Property introspection utilities initialized (Data Source aware)")
logger.info("Functions: introspect_database_properties(), get_property_metadata(), safe_property_value()")


2026-01-27 15:51:51 [INFO] Property introspection utilities initialized (Data Source aware)
2026-01-27 15:51:51 [INFO] Functions: introspect_database_properties(), get_property_metadata(), safe_property_value()


In [8]:
# ============================================================
# Cell 08 — Papers DB CRUD wrappers (researchOS schema + Data Source query)
# ============================================================
# Overview:
#   CRUD wrappers for Papers DB aligned to current researchOS Notion schema:
#   - Name (title)
#   - Authors & Year (rich_text)
#   - Tags (multi_select)
#   - PDF Link (url)
#   - Findings / Core Idea / Notes / Methods / Datasets
#   - Type / Source (select)
#   - Papers (relation)
#   Plus Daily-minimum additions (Plan A):
#   - Status (select), Dedup Key (rich_text), Source UID (rich_text),
#     Ingested At (date), Run ID (rich_text), PDF Status (select), Slide 1 URL (url)
#
# Notes:
#   - Query uses /data_sources/{id}/query (Notion 2025+)
#   - Create/Update uses /pages (unchanged)
#   - Property construction should use safe_property_value() when possible
#

from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime, timezone

PAPERS_DB = PAPERS_DB_ID

# --- Property names (single source of truth) ---
PAPER_PROPS = {
    "name": "Name",
    "created_time": "Created time",
    "authors_year": "Authors & Year",
    "tags": "Tags",
    "pdf_link": "PDF Link",
    "findings": "Findings",
    "core_idea": "Core Idea",
    "notes": "Notes",
    "methods": "Methods",
    "type": "Type",
    "source": "Source",
    "datasets": "Datasets",
    "papers_rel": "Papers",
    # Plan A additions
    "status": "Status",
    "dedup_key": "Dedup Key",
    "source_uid": "Source UID",
    "ingested_at": "Ingested At",
    "run_id": "Run ID",
    "pdf_status": "PDF Status",
    "slide1_url": "Slide 1 URL",
}

# --- Helper: Data Source query wrapper (if not already defined elsewhere) ---
def query_database(database_id: str, payload: dict) -> dict:
    ds_id = get_primary_data_source_id(database_id)
    return notion_client.request("POST", f"/data_sources/{ds_id}/query", json=payload)

# --- Helper: extract plain text from title/rich_text ---
def _extract_plain_text(rich_list: list) -> str:
    if not rich_list:
        return ""
    # Notion returns items with plain_text
    return "".join([x.get("plain_text", "") for x in rich_list if isinstance(x, dict)])

# --- Extract paper properties from a Notion page ---
def extract_paper_properties(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties", {}) or {}

    out = {
        "page_id": page.get("id"),
        "created_time": page.get("created_time"),
        "last_edited_time": page.get("last_edited_time"),
        "url": page.get("url"),
    }

    # Name (title)
    name_obj = props.get(PAPER_PROPS["name"], {}) or {}
    out["name"] = _extract_plain_text(name_obj.get("title", []) or [])

    # Authors & Year
    ay_obj = props.get(PAPER_PROPS["authors_year"], {}) or {}
    out["authors_year"] = _extract_plain_text(ay_obj.get("rich_text", []) or [])

    # Tags (multi_select)
    tags_obj = props.get(PAPER_PROPS["tags"], {}) or {}
    out["tags"] = [t.get("name") for t in (tags_obj.get("multi_select", []) or []) if t.get("name")]

    # PDF Link (url)
    pdf_obj = props.get(PAPER_PROPS["pdf_link"], {}) or {}
    out["pdf_link"] = pdf_obj.get("url")

    # Status (select)
    status_obj = props.get(PAPER_PROPS["status"], {}) or {}
    out["status"] = (status_obj.get("select") or {}).get("name")

    # PDF Status (select)
    pdfs_obj = props.get(PAPER_PROPS["pdf_status"], {}) or {}
    out["pdf_status"] = (pdfs_obj.get("select") or {}).get("name")

    # Dedup Key / Source UID / Run ID
    dk_obj = props.get(PAPER_PROPS["dedup_key"], {}) or {}
    out["dedup_key"] = _extract_plain_text(dk_obj.get("rich_text", []) or [])

    su_obj = props.get(PAPER_PROPS["source_uid"], {}) or {}
    out["source_uid"] = _extract_plain_text(su_obj.get("rich_text", []) or [])

    rid_obj = props.get(PAPER_PROPS["run_id"], {}) or {}
    out["run_id"] = _extract_plain_text(rid_obj.get("rich_text", []) or [])

    # Ingested At (date)
    ia_obj = props.get(PAPER_PROPS["ingested_at"], {}) or {}
    out["ingested_at"] = (ia_obj.get("date") or {}).get("start")

    # Slide 1 URL (url)
    s1_obj = props.get(PAPER_PROPS["slide1_url"], {}) or {}
    out["slide1_url"] = s1_obj.get("url")

    return out


# --- Dedup: query by Dedup Key (recommended) ---
def query_paper_by_dedup_key(dedup_key: str) -> Optional[Dict[str, Any]]:
    if not dedup_key:
        return None

    payload = {
        "page_size": 1,
        "filter": {
            "property": PAPER_PROPS["dedup_key"],
            "rich_text": {"equals": dedup_key},
        },
    }
    res = query_database(PAPERS_DB, payload)
    results = res.get("results", []) or []
    if not results:
        return None
    return extract_paper_properties(results[0])


# --- Create paper (minimal fields to support Daily inbox) ---
def create_paper_inbox(
    name: str,
    authors_year: str = "",
    pdf_link: Optional[str] = None,
    tags: Optional[List[str]] = None,
    status: str = "INBOX",                 # ← DBにStatusが無いなら自然に無視されます
    pdf_status: str = "NONE",              # ← スクショでは "PDF Status" がある（LOCAL等）
    dedup_key: Optional[str] = None,
    source_uid: Optional[str] = None,      # ← DBに無いなら無視されます
    run_id: Optional[str] = None,
    slide1_url: Optional[str] = None,
    extra: Optional[Dict[str, Any]] = None,  # may include notion_fields etc.
) -> Dict[str, Any]:
    """
    Create a new paper entry aligned to the *actual* Notion schema shown in the screenshot.

    Expected property names (if present):
      - Title property (auto-detected)
      - Authors & Year
      - Core Idea, Datasets, Findings, Methods, Notes
      - PDF Link, Source, Tags, Type
      - Papers (relation)
      - Dedup Key, Ingested At, PDF Status, Run ID, Slide 1 URL
    """
    if not name or not str(name).strip():
        raise ValueError("name is required")

    now_iso = datetime.now(timezone.utc).date().isoformat()

    # Introspect types once (cached)
    types = introspect_database_properties(PAPERS_DB, use_cache=True) or {}

    # --- helpers ---
    def _first_title_prop_name() -> str:
        # Find the title property in this DB (robust)
        for prop_name, meta in types.items():
            try:
                t = meta.get("type") if isinstance(meta, dict) else None
            except Exception:
                t = None
            if t == "title":
                return prop_name
        # fallback to common names
        for cand in ["Name", "Title"]:
            if cand in types:
                return cand
        # if none found, we can't create a page safely
        raise RuntimeError("No title property found in Papers DB schema (cannot create page).")

    TITLE_PROP = _first_title_prop_name()

    # Exact property names (match screenshot)
    P = {
        "AUTHORS_YEAR": "Authors & Year",
        "CORE_IDEA": "Core Idea",
        "DATASETS": "Datasets",
        "FINDINGS": "Findings",
        "METHODS": "Methods",
        "NOTES": "Notes",
        "PDF_LINK": "PDF Link",
        "SOURCE": "Source",
        "TAGS": "Tags",
        "TYPE": "Type",
        "PAPERS_REL": "Papers",
        "DEDUP_KEY": "Dedup Key",
        "INGESTED_AT": "Ingested At",
        "PDF_STATUS": "PDF Status",
        "RUN_ID": "Run ID",
        "SLIDE1_URL": "Slide 1 URL",
        # optional / may exist
        "STATUS": "Status",
        "SOURCE_UID": "Source UID",
    }

    notion_props: Dict[str, Any] = {}

    def _set_if_exists(prop_name: str, value: Any):
        if value is None:
            return
        if isinstance(value, str) and value.strip() == "":
            return
        if prop_name not in types:
            return
        notion_props[prop_name] = safe_property_value(types[prop_name], value, prop_name)

    # --- required title ---
    notion_props[TITLE_PROP] = safe_property_value(types[TITLE_PROP], str(name).strip(), TITLE_PROP)

    # --- baseline fields (only if present) ---
    _set_if_exists(P["AUTHORS_YEAR"], authors_year)

    if tags is not None:
        clean_tags = [str(t).strip() for t in (tags or []) if str(t).strip()]
        _set_if_exists(P["TAGS"], clean_tags if clean_tags else None)

    _set_if_exists(P["PDF_LINK"], pdf_link)
    _set_if_exists(P["PDF_STATUS"], pdf_status)
    _set_if_exists(P["DEDUP_KEY"], dedup_key)
    _set_if_exists(P["RUN_ID"], run_id)
    _set_if_exists(P["INGESTED_AT"], now_iso)
    _set_if_exists(P["SLIDE1_URL"], slide1_url)

    # Optional if your DB has them
    _set_if_exists(P.get("STATUS"), status)
    _set_if_exists(P.get("SOURCE_UID"), source_uid)

    # --- self-healing notion_fields (from extra.notioon_fields) ---
    notion_fields = None
    if isinstance(extra, dict):
        nf = extra.get("notion_fields")
        if isinstance(nf, dict) and nf:
            notion_fields = nf

    if notion_fields:
        # English fields
        if isinstance(notion_fields.get("source"), str):
            _set_if_exists(P["SOURCE"], notion_fields["source"].strip())
        if isinstance(notion_fields.get("type"), str):
            _set_if_exists(P["TYPE"], notion_fields["type"].strip())

        nf_tags = notion_fields.get("tags")
        if isinstance(nf_tags, list):
            clean = [str(t).strip() for t in nf_tags if str(t).strip()]
            if clean:
                _set_if_exists(P["TAGS"], clean)

        # Japanese rich fields
        for k, prop in [
            ("core_idea", P["CORE_IDEA"]),
            ("datasets",  P["DATASETS"]),
            ("methods",   P["METHODS"]),
            ("findings",  P["FINDINGS"]),
            ("notes",     P["NOTES"]),
        ]:
            v = notion_fields.get(k)
            if isinstance(v, str) and v.strip():
                _set_if_exists(prop, v.strip())

    # --- allow extra to directly set real Notion props by name ---
    # e.g. extra["Core Idea"]="..."  / extra["Papers"]=[page_ids] etc.
    if isinstance(extra, dict):
        for prop_name, v in extra.items():
            if prop_name == "notion_fields":
                continue
            if prop_name not in types:
                # only warn for “looks intended for DB”
                if prop_name in set(P.values()):
                    logger.warning("Extra property not found in schema: %s (skipping)", prop_name)
                continue
            try:
                # If caller passes already-formatted Notion prop object, allow it:
                # e.g. {"rich_text":[...]} / {"relation":[...]} etc.
                if isinstance(v, dict) and ("type" in v or any(k in v for k in ["rich_text","title","select","multi_select","url","date","relation"])):
                    notion_props[prop_name] = v
                else:
                    notion_props[prop_name] = safe_property_value(types[prop_name], v, prop_name)
            except Exception as e:
                logger.warning("Failed to set extra prop %s: %s (skipping)", prop_name, e)

    payload = {
        "parent": {"database_id": PAPERS_DB},
        "properties": notion_props,
    }

    page = notion_client.request("POST", "/pages", json=payload)
    logger.info("Created paper: %s", name)
    return extract_paper_properties(page)



# --- Update paper ---
def update_paper(page_id: str, updates: Dict[str, Any]) -> Dict[str, Any]:
    """
    updates: dict of logical keys -> values (by Notion property name)
    Example: {"Status": "DONE", "Notes": "..." }
    """
    if not page_id:
        raise ValueError("page_id is required")

    types = introspect_database_properties(PAPERS_DB, use_cache=True)

    notion_props: Dict[str, Any] = {}
    for prop_name, value in updates.items():
        if prop_name not in types:
            logger.warning("Unknown property '%s' (skipping)", prop_name)
            continue
        notion_props[prop_name] = safe_property_value(types[prop_name], value, prop_name)

    payload = {"properties": notion_props}
    page = notion_client.request("PATCH", f"/pages/{page_id}", json=payload)
    logger.info("Updated paper page: %s", page_id)
    return extract_paper_properties(page)


# --- Get recent papers (by Ingested At if present, else Created time) ---
def get_recent_papers(limit: int = 10, status_filter: Optional[str] = None) -> List[Dict[str, Any]]:
    types = introspect_database_properties(PAPERS_DB, use_cache=True)

    sort_prop = PAPER_PROPS["ingested_at"] if PAPER_PROPS["ingested_at"] in types else PAPER_PROPS["created_time"]

    payload: Dict[str, Any] = {
        "page_size": min(limit, 100),
        "sorts": [{"property": sort_prop, "direction": "descending"}],
    }

    if status_filter and PAPER_PROPS["status"] in types:
        payload["filter"] = {
            "property": PAPER_PROPS["status"],
            "select": {"equals": status_filter},
        }

    res = query_database(PAPERS_DB, payload)
    results = res.get("results", []) or []
    return [extract_paper_properties(p) for p in results]


# --- Dedup helper ---
def check_duplicate_paper(dedup_key: Optional[str], name: str) -> Tuple[bool, Optional[str]]:
    if dedup_key:
        existing = query_paper_by_dedup_key(dedup_key)
        if existing:
            return True, existing["page_id"]

    # fallback: simple name contains search
    payload = {
        "page_size": 5,
        "filter": {"property": PAPER_PROPS["name"], "title": {"contains": name}},
    }
    res = query_database(PAPERS_DB, payload)
    results = res.get("results", []) or []
    for p in results:
        ex = extract_paper_properties(p)
        if ex.get("name", "").strip().lower() == name.strip().lower():
            return True, ex["page_id"]

    return False, None


logger.info("Papers DB CRUD wrappers initialized (researchOS schema + data_sources query)")


2026-01-27 15:51:58 [INFO] Papers DB CRUD wrappers initialized (researchOS schema + data_sources query)


In [9]:
# ============================================================
# Cell 09 — Events DB CRUD wrappers (researchOS schema + Data Source query)
# ============================================================
# Overview:
#   CRUD wrappers for Events DB aligned to current researchOS Notion schema:
#   Required:
#     - Name (title)
#     - Date (date)
#     - Detected At (date)
#     - Target (relation -> Monitoring Targets)
#     - Event Type (select)
#     - Source URL (url)
#     - Source (select)
#     - Summary (rich_text)
#     - Confidence (number)
#     - Dedup Key (rich_text)
#     - Status (select)
#   Plan A additions:
#     - Run ID (rich_text)
#     - Ingested At (date)
#     - Action Needed (checkbox)
#     - Related Papers (relation -> Papers)
#
# Notes:
#   - Query uses /data_sources/{id}/query (Notion 2025+)
#   - Create/Update uses /pages
#

from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime, timezone

EVENTS_DB = EVENTS_DB_ID

EVENT_PROPS = {
    "name": "Name",
    "date": "Date",
    "detected_at": "Detected At",
    "target": "Target",
    "event_type": "Event Type",
    "source_url": "Source URL",
    "source": "Source",
    "summary": "Summary",
    "confidence": "Confidence",
    "dedup_key": "Dedup Key",
    "status": "Status",
    # Plan A
    "run_id": "Run ID",
    "ingested_at": "Ingested At",
    "action_needed": "Action Needed",
    "related_papers": "Related Papers",
}

def query_database(database_id: str, payload: dict) -> dict:
    ds_id = get_primary_data_source_id(database_id)
    return notion_client.request("POST", f"/data_sources/{ds_id}/query", json=payload)

def _extract_plain_text(rich_list: list) -> str:
    if not rich_list:
        return ""
    return "".join([x.get("plain_text", "") for x in rich_list if isinstance(x, dict)])

def extract_event_properties(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties", {}) or {}
    out = {
        "page_id": page.get("id"),
        "url": page.get("url"),
        "created_time": page.get("created_time"),
        "last_edited_time": page.get("last_edited_time"),
    }

    # Name (title)
    n = props.get(EVENT_PROPS["name"], {}) or {}
    out["name"] = _extract_plain_text(n.get("title", []) or [])

    # Date fields
    for k in ("date", "detected_at", "ingested_at"):
        pn = EVENT_PROPS.get(k)
        if not pn:
            continue
        dobj = props.get(pn, {}) or {}
        out[k] = (dobj.get("date") or {}).get("start")

    # Selects
    for k in ("event_type", "source", "status"):
        pn = EVENT_PROPS[k]
        sobj = props.get(pn, {}) or {}
        out[k] = (sobj.get("select") or {}).get("name")

    # URL
    uobj = props.get(EVENT_PROPS["source_url"], {}) or {}
    out["source_url"] = uobj.get("url") or ""

    # Summary
    s = props.get(EVENT_PROPS["summary"], {}) or {}
    out["summary"] = _extract_plain_text(s.get("rich_text", []) or [])

    # Confidence
    c = props.get(EVENT_PROPS["confidence"], {}) or {}
    out["confidence"] = c.get("number")

    # Dedup Key / Run ID
    dk = props.get(EVENT_PROPS["dedup_key"], {}) or {}
    out["dedup_key"] = _extract_plain_text(dk.get("rich_text", []) or [])

    rid = props.get(EVENT_PROPS["run_id"], {}) or {}
    out["run_id"] = _extract_plain_text(rid.get("rich_text", []) or [])

    # Action Needed
    an = props.get(EVENT_PROPS["action_needed"], {}) or {}
    out["action_needed"] = an.get("checkbox")

    # Relations (Target, Related Papers)
    t = props.get(EVENT_PROPS["target"], {}) or {}
    out["target_ids"] = [r.get("id") for r in (t.get("relation", []) or []) if r.get("id")]

    rp = props.get(EVENT_PROPS["related_papers"], {}) or {}
    out["related_paper_ids"] = [r.get("id") for r in (rp.get("relation", []) or []) if r.get("id")]

    return out


def query_event_by_dedup_key(dedup_key: str) -> Optional[Dict[str, Any]]:
    if not dedup_key:
        return None
    payload = {
        "page_size": 1,
        "filter": {
            "property": EVENT_PROPS["dedup_key"],
            "rich_text": {"equals": dedup_key},
        },
    }
    res = query_database(EVENTS_DB, payload)
    results = res.get("results", []) or []
    if not results:
        return None
    return extract_event_properties(results[0])


def create_event(
    name: str,
    date: str,
    detected_at: Optional[str],
    target_page_ids: List[str],
    event_type: str,
    source_url: str,
    source: str,
    summary: str,
    confidence: Optional[float] = None,
    status: str = "NEW",
    dedup_key: Optional[str] = None,
    run_id: Optional[str] = None,
    action_needed: bool = False,
    related_paper_ids: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """
    Create new Event aligned to current researchOS schema.
    """
    if not name:
        raise ValueError("name is required")
    if not date:
        raise ValueError("date is required (YYYY-MM-DD)")
    if not target_page_ids:
        raise ValueError("target_page_ids is required (relation)")
    if not event_type:
        raise ValueError("event_type is required")
    if not source_url:
        raise ValueError("source_url is required")
    if not source:
        raise ValueError("source is required")
    if summary is None:
        summary = ""

    now_iso = datetime.now(timezone.utc).date().isoformat()
    detected_at = detected_at or now_iso

    # Introspect property types for safe payload construction
    types = introspect_database_properties(EVENTS_DB, use_cache=True)

    props: Dict[str, Any] = {}
    props[EVENT_PROPS["name"]] = safe_property_value(types[EVENT_PROPS["name"]], name, EVENT_PROPS["name"])
    props[EVENT_PROPS["date"]] = safe_property_value(types[EVENT_PROPS["date"]], date, EVENT_PROPS["date"])
    props[EVENT_PROPS["detected_at"]] = safe_property_value(types[EVENT_PROPS["detected_at"]], detected_at, EVENT_PROPS["detected_at"])
    props[EVENT_PROPS["target"]] = safe_property_value(types[EVENT_PROPS["target"]], target_page_ids, EVENT_PROPS["target"])
    props[EVENT_PROPS["event_type"]] = safe_property_value(types[EVENT_PROPS["event_type"]], event_type, EVENT_PROPS["event_type"])
    props[EVENT_PROPS["source_url"]] = safe_property_value(types[EVENT_PROPS["source_url"]], source_url, EVENT_PROPS["source_url"])
    props[EVENT_PROPS["source"]] = safe_property_value(types[EVENT_PROPS["source"]], source, EVENT_PROPS["source"])
    props[EVENT_PROPS["summary"]] = safe_property_value(types[EVENT_PROPS["summary"]], summary, EVENT_PROPS["summary"])

    if confidence is not None and EVENT_PROPS["confidence"] in types:
        props[EVENT_PROPS["confidence"]] = safe_property_value(types[EVENT_PROPS["confidence"]], float(confidence), EVENT_PROPS["confidence"])
    elif EVENT_PROPS["confidence"] in types:
        props[EVENT_PROPS["confidence"]] = safe_property_value(types[EVENT_PROPS["confidence"]], None, EVENT_PROPS["confidence"])

    if EVENT_PROPS["status"] in types:
        props[EVENT_PROPS["status"]] = safe_property_value(types[EVENT_PROPS["status"]], status, EVENT_PROPS["status"])

    if dedup_key and EVENT_PROPS["dedup_key"] in types:
        props[EVENT_PROPS["dedup_key"]] = safe_property_value(types[EVENT_PROPS["dedup_key"]], dedup_key, EVENT_PROPS["dedup_key"])

    if run_id and EVENT_PROPS["run_id"] in types:
        props[EVENT_PROPS["run_id"]] = safe_property_value(types[EVENT_PROPS["run_id"]], run_id, EVENT_PROPS["run_id"])

    if EVENT_PROPS["ingested_at"] in types:
        props[EVENT_PROPS["ingested_at"]] = safe_property_value(types[EVENT_PROPS["ingested_at"]], now_iso, EVENT_PROPS["ingested_at"])

    if EVENT_PROPS["action_needed"] in types:
        props[EVENT_PROPS["action_needed"]] = safe_property_value(types[EVENT_PROPS["action_needed"]], bool(action_needed), EVENT_PROPS["action_needed"])

    if related_paper_ids and EVENT_PROPS["related_papers"] in types:
        props[EVENT_PROPS["related_papers"]] = safe_property_value(types[EVENT_PROPS["related_papers"]], related_paper_ids, EVENT_PROPS["related_papers"])

    payload = {"parent": {"database_id": EVENTS_DB}, "properties": props}
    page = notion_client.request("POST", "/pages", json=payload)
    logger.info("Created event: %s", name)
    return extract_event_properties(page)


def update_event(page_id: str, updates: Dict[str, Any]) -> Dict[str, Any]:
    """
    updates: {NotionPropertyName: python_value}
    Example: {"Status": "DONE", "Action Needed": True}
    """
    if not page_id:
        raise ValueError("page_id is required")

    types = introspect_database_properties(EVENTS_DB, use_cache=True)

    notion_props: Dict[str, Any] = {}
    for prop_name, value in updates.items():
        if prop_name not in types:
            logger.warning("Unknown property '%s' (skipping)", prop_name)
            continue
        notion_props[prop_name] = safe_property_value(types[prop_name], value, prop_name)

    page = notion_client.request("PATCH", f"/pages/{page_id}", json={"properties": notion_props})
    logger.info("Updated event page: %s", page_id)
    return extract_event_properties(page)


def get_recent_events(limit: int = 10, status_filter: Optional[str] = None) -> List[Dict[str, Any]]:
    types = introspect_database_properties(EVENTS_DB, use_cache=True)
    sort_prop = EVENT_PROPS["ingested_at"] if EVENT_PROPS["ingested_at"] in types else EVENT_PROPS["detected_at"]

    payload: Dict[str, Any] = {
        "page_size": min(limit, 100),
        "sorts": [{"property": sort_prop, "direction": "descending"}],
    }

    if status_filter and EVENT_PROPS["status"] in types:
        payload["filter"] = {"property": EVENT_PROPS["status"], "select": {"equals": status_filter}}

    res = query_database(EVENTS_DB, payload)
    results = res.get("results", []) or []
    return [extract_event_properties(p) for p in results]


logger.info("Events DB CRUD wrappers initialized (researchOS schema + data_sources query)")


2026-01-27 15:52:14 [INFO] Events DB CRUD wrappers initialized (researchOS schema + data_sources query)


In [10]:
# ============================================================
# Cell 10 — Monitoring Targets DB CRUD wrappers (researchOS schema + Plan A)
# ============================================================
# Overview:
#   CRUD wrappers for Monitoring Targets aligned to current researchOS schema:
#   Required:
#     - Name (title)
#     - Type (select)
#     - Status (select)
#     - Priority (select or number; treat as select by default)
#     - Search Keywords (rich_text)
#     - Source URLs (rich_text)
#     - Cadence (select)
#     - Last Checked (date)
#     - Next Check (date)
#   Plan A additions:
#     - Enabled (checkbox)
#     - Source Type (select)
#     - Last Error (rich_text)
#     - Error Count (number)
#
# Notes:
#   - Query uses /data_sources/{id}/query
#   - Create/Update uses /pages
#

from typing import Dict, Any, Optional, List
from datetime import datetime, timezone

MONITORING_TARGETS_DB = MONITORING_TARGETS_DB_ID

TARGET_PROPS = {
    "name": "Name",
    "type": "Type",
    "status": "Status",
    "priority": "Priority",
    "search_keywords": "Search Keywords",
    "source_urls": "Source URLs",
    "cadence": "Cadence",
    "last_checked": "Last Checked",
    "next_check": "Next Check",
    # Plan A
    "enabled": "Enabled",
    "source_type": "Source Type",
    "last_error": "Last Error",
    "error_count": "Error Count",
}

def query_database(database_id: str, payload: dict) -> dict:
    ds_id = get_primary_data_source_id(database_id)
    return notion_client.request("POST", f"/data_sources/{ds_id}/query", json=payload)

def _extract_plain_text(rich_list: list) -> str:
    if not rich_list:
        return ""
    return "".join([x.get("plain_text", "") for x in rich_list if isinstance(x, dict)])

def extract_monitoring_target_properties(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties", {}) or {}
    out = {
        "page_id": page.get("id"),
        "url": page.get("url"),
        "created_time": page.get("created_time"),
        "last_edited_time": page.get("last_edited_time"),
    }

    # Name
    n = props.get(TARGET_PROPS["name"], {}) or {}
    out["name"] = _extract_plain_text(n.get("title", []) or [])

    # selects
    for k in ("type", "status", "cadence", "source_type"):
        pn = TARGET_PROPS.get(k)
        if not pn:
            continue
        sobj = props.get(pn, {}) or {}
        out[k] = (sobj.get("select") or {}).get("name")

    # Priority: could be select or number depending on your DB
    pr = props.get(TARGET_PROPS["priority"], {}) or {}
    if pr.get("type") == "number" or "number" in pr:
        out["priority"] = pr.get("number")
    else:
        out["priority"] = (pr.get("select") or {}).get("name")

    # Search Keywords / Source URLs (rich_text)
    sk = props.get(TARGET_PROPS["search_keywords"], {}) or {}
    out["search_keywords"] = _extract_plain_text(sk.get("rich_text", []) or [])

    su = props.get(TARGET_PROPS["source_urls"], {}) or {}
    out["source_urls"] = _extract_plain_text(su.get("rich_text", []) or [])

    # dates
    for k in ("last_checked", "next_check"):
        pn = TARGET_PROPS[k]
        dobj = props.get(pn, {}) or {}
        out[k] = (dobj.get("date") or {}).get("start")

    # Enabled
    en = props.get(TARGET_PROPS["enabled"], {}) or {}
    out["enabled"] = en.get("checkbox")

    # Last Error
    le = props.get(TARGET_PROPS["last_error"], {}) or {}
    out["last_error"] = _extract_plain_text(le.get("rich_text", []) or [])

    # Error Count
    ec = props.get(TARGET_PROPS["error_count"], {}) or {}
    out["error_count"] = ec.get("number")

    return out


def create_monitoring_target(
    name: str,
    target_type: str,
    status: str,
    priority: Any,
    search_keywords: str,
    source_urls: str,
    cadence: str,
    next_check: str,
    enabled: bool = True,
    source_type: Optional[str] = None,
    run_id: Optional[str] = None,  # optional: if you later add "Last Run ID" etc.
) -> Dict[str, Any]:
    """
    Create a Monitoring Target aligned to current researchOS schema.
    """
    if not name:
        raise ValueError("name is required")
    if not target_type:
        raise ValueError("target_type is required")
    if not status:
        raise ValueError("status is required")
    if search_keywords is None:
        search_keywords = ""
    if source_urls is None:
        source_urls = ""
    if not cadence:
        raise ValueError("cadence is required")
    if not next_check:
        raise ValueError("next_check is required (YYYY-MM-DD)")

    today_iso = datetime.now(timezone.utc).date().isoformat()

    types = introspect_database_properties(MONITORING_TARGETS_DB, use_cache=True)

    props: Dict[str, Any] = {}
    props[TARGET_PROPS["name"]] = safe_property_value(types[TARGET_PROPS["name"]], name, TARGET_PROPS["name"])
    props[TARGET_PROPS["type"]] = safe_property_value(types[TARGET_PROPS["type"]], target_type, TARGET_PROPS["type"])
    props[TARGET_PROPS["status"]] = safe_property_value(types[TARGET_PROPS["status"]], status, TARGET_PROPS["status"])

    # Priority may be select or number
    pr_name = TARGET_PROPS["priority"]
    if pr_name in types:
        props[pr_name] = safe_property_value(types[pr_name], priority, pr_name)

    props[TARGET_PROPS["search_keywords"]] = safe_property_value(types[TARGET_PROPS["search_keywords"]], search_keywords, TARGET_PROPS["search_keywords"])
    props[TARGET_PROPS["source_urls"]] = safe_property_value(types[TARGET_PROPS["source_urls"]], source_urls, TARGET_PROPS["source_urls"])
    props[TARGET_PROPS["cadence"]] = safe_property_value(types[TARGET_PROPS["cadence"]], cadence, TARGET_PROPS["cadence"])

    # dates
    props[TARGET_PROPS["last_checked"]] = safe_property_value(types[TARGET_PROPS["last_checked"]], None, TARGET_PROPS["last_checked"])
    props[TARGET_PROPS["next_check"]] = safe_property_value(types[TARGET_PROPS["next_check"]], next_check, TARGET_PROPS["next_check"])

    # Plan A
    if TARGET_PROPS["enabled"] in types:
        props[TARGET_PROPS["enabled"]] = safe_property_value(types[TARGET_PROPS["enabled"]], bool(enabled), TARGET_PROPS["enabled"])
    if source_type and TARGET_PROPS["source_type"] in types:
        props[TARGET_PROPS["source_type"]] = safe_property_value(types[TARGET_PROPS["source_type"]], source_type, TARGET_PROPS["source_type"])
    if TARGET_PROPS["last_error"] in types:
        props[TARGET_PROPS["last_error"]] = safe_property_value(types[TARGET_PROPS["last_error"]], "", TARGET_PROPS["last_error"])
    if TARGET_PROPS["error_count"] in types:
        props[TARGET_PROPS["error_count"]] = safe_property_value(types[TARGET_PROPS["error_count"]], 0, TARGET_PROPS["error_count"])

    payload = {"parent": {"database_id": MONITORING_TARGETS_DB}, "properties": props}
    page = notion_client.request("POST", "/pages", json=payload)
    logger.info("Created monitoring target: %s", name)
    return extract_monitoring_target_properties(page)


def get_enabled_targets() -> List[Dict[str, Any]]:
    """
    Retrieve targets where Enabled == True.
    """
    types = introspect_database_properties(MONITORING_TARGETS_DB, use_cache=True)

    payload: Dict[str, Any] = {"page_size": 100}
    if TARGET_PROPS["enabled"] in types:
        payload["filter"] = {"property": TARGET_PROPS["enabled"], "checkbox": {"equals": True}}

    res = query_database(MONITORING_TARGETS_DB, payload)
    results = res.get("results", []) or []
    return [extract_monitoring_target_properties(p) for p in results]


def update_target(page_id: str, updates: Dict[str, Any]) -> Dict[str, Any]:
    """
    updates: {NotionPropertyName: python_value}
    Example: {"Last Checked": "2026-01-25", "Next Check": "2026-01-26"}
    """
    if not page_id:
        raise ValueError("page_id is required")

    types = introspect_database_properties(MONITORING_TARGETS_DB, use_cache=True)

    notion_props: Dict[str, Any] = {}
    for prop_name, value in updates.items():
        if prop_name not in types:
            logger.warning("Unknown property '%s' (skipping)", prop_name)
            continue
        notion_props[prop_name] = safe_property_value(types[prop_name], value, prop_name)

    page = notion_client.request("PATCH", f"/pages/{page_id}", json={"properties": notion_props})
    logger.info("Updated target: %s", page_id)
    return extract_monitoring_target_properties(page)


def mark_target_checked(page_id: str, checked_date: Optional[str] = None, next_check: Optional[str] = None) -> Dict[str, Any]:
    """
    Convenience helper to update Last Checked (+ optionally Next Check).
    """
    checked_date = checked_date or datetime.now(timezone.utc).date().isoformat()
    updates = {TARGET_PROPS["last_checked"]: checked_date}
    if next_check:
        updates[TARGET_PROPS["next_check"]] = next_check
    return update_target(page_id, updates)


def record_target_error(page_id: str, error_message: str) -> Dict[str, Any]:
    """
    Increment Error Count and set Last Error.
    (Reads are avoided; we just set Last Error and rely on separate monitoring to count if needed.)
    """
    # If you want strict increment, you need a read first. Keep it simple for now.
    updates = {}
    updates[TARGET_PROPS["last_error"]] = (error_message or "")[:2000]
    return update_target(page_id, updates)


logger.info("Monitoring Targets DB CRUD wrappers initialized (researchOS schema + Plan A)")


2026-01-27 15:52:23 [INFO] Monitoring Targets DB CRUD wrappers initialized (researchOS schema + Plan A)


In [11]:
# ============================================================
# Cell 11 — Monitoring Queue DB CRUD wrappers (researchOS schema + Plan A)
# ============================================================
# Overview:
#   CRUD wrappers for Monitoring Queue aligned to current researchOS schema:
#   Required:
#     - Name (title)
#     - Queue Type (select)
#     - Target (relation -> Monitoring Targets)
#     - Scheduled At (date)
#     - Status (select)
#     - Attempts (number)
#     - Max Attempts (number)
#     - Last Error (rich_text)
#     - Run ID (rich_text)
#   Plan A additions:
#     - Dedup Key (rich_text)
#     - Payload (rich_text)
#     - Last Attempt At (date)
#     - Result ID (rich_text)
#     - Result URL (url)
#
# Notes:
#   - Query uses /data_sources/{id}/query
#   - Create/Update uses /pages
#   - Payload is stored as rich_text (JSON string recommended; keep small)
#

from typing import Dict, Any, Optional, List
from datetime import datetime, timezone
import json as _json

MONITORING_QUEUE_DB = MONITORING_QUEUE_DB_ID

QUEUE_PROPS = {
    "name": "Name",
    "queue_type": "Queue Type",
    "target": "Target",
    "scheduled_at": "Scheduled At",
    "status": "Status",
    "attempts": "Attempts",
    "max_attempts": "Max Attempts",
    "last_error": "Last Error",
    "run_id": "Run ID",
    # Plan A
    "dedup_key": "Dedup Key",
    "payload": "Payload",
    "last_attempt_at": "Last Attempt At",
    "result_id": "Result ID",
    "result_url": "Result URL",
}

def query_database(database_id: str, payload: dict) -> dict:
    ds_id = get_primary_data_source_id(database_id)
    return notion_client.request("POST", f"/data_sources/{ds_id}/query", json=payload)

def _extract_plain_text(rich_list: list) -> str:
    if not rich_list:
        return ""
    return "".join([x.get("plain_text", "") for x in rich_list if isinstance(x, dict)])

def extract_queue_item_properties(page: Dict[str, Any]) -> Dict[str, Any]:
    props = page.get("properties", {}) or {}
    out = {
        "page_id": page.get("id"),
        "url": page.get("url"),
        "created_time": page.get("created_time"),
        "last_edited_time": page.get("last_edited_time"),
    }

    # Name
    n = props.get(QUEUE_PROPS["name"], {}) or {}
    out["name"] = _extract_plain_text(n.get("title", []) or [])

    # Selects
    for k in ("queue_type", "status"):
        pn = QUEUE_PROPS[k]
        sobj = props.get(pn, {}) or {}
        out[k] = (sobj.get("select") or {}).get("name")

    # Relations
    t = props.get(QUEUE_PROPS["target"], {}) or {}
    out["target_ids"] = [r.get("id") for r in (t.get("relation", []) or []) if r.get("id")]

    # Numbers
    a = props.get(QUEUE_PROPS["attempts"], {}) or {}
    out["attempts"] = a.get("number")
    ma = props.get(QUEUE_PROPS["max_attempts"], {}) or {}
    out["max_attempts"] = ma.get("number")

    # Dates
    for k in ("scheduled_at", "last_attempt_at"):
        pn = QUEUE_PROPS[k]
        dobj = props.get(pn, {}) or {}
        out[k] = (dobj.get("date") or {}).get("start")

    # Text fields
    le = props.get(QUEUE_PROPS["last_error"], {}) or {}
    out["last_error"] = _extract_plain_text(le.get("rich_text", []) or [])

    rid = props.get(QUEUE_PROPS["run_id"], {}) or {}
    out["run_id"] = _extract_plain_text(rid.get("rich_text", []) or [])

    dk = props.get(QUEUE_PROPS["dedup_key"], {}) or {}
    out["dedup_key"] = _extract_plain_text(dk.get("rich_text", []) or [])

    pl = props.get(QUEUE_PROPS["payload"], {}) or {}
    out["payload"] = _extract_plain_text(pl.get("rich_text", []) or [])

    res_id = props.get(QUEUE_PROPS["result_id"], {}) or {}
    out["result_id"] = _extract_plain_text(res_id.get("rich_text", []) or [])

    res_url = props.get(QUEUE_PROPS["result_url"], {}) or {}
    out["result_url"] = res_url.get("url")

    return out


def create_queue_item(
    name: str,
    queue_type: str,
    target_page_ids: List[str],
    scheduled_at: str,
    status: str = "NEW",
    attempts: int = 0,
    max_attempts: int = 3,
    run_id: Optional[str] = None,
    dedup_key: Optional[str] = None,
    payload: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Create a queue item (researchOS schema).
    payload: dict -> JSON string stored into "Payload" (rich_text).
    """
    if not name:
        raise ValueError("name is required")
    if not queue_type:
        raise ValueError("queue_type is required")
    if not target_page_ids:
        raise ValueError("target_page_ids is required (relation)")
    if not scheduled_at:
        raise ValueError("scheduled_at is required (YYYY-MM-DD or ISO datetime)")

    types = introspect_database_properties(MONITORING_QUEUE_DB, use_cache=True)

    props: Dict[str, Any] = {}
    props[QUEUE_PROPS["name"]] = safe_property_value(types[QUEUE_PROPS["name"]], name, QUEUE_PROPS["name"])
    props[QUEUE_PROPS["queue_type"]] = safe_property_value(types[QUEUE_PROPS["queue_type"]], queue_type, QUEUE_PROPS["queue_type"])
    props[QUEUE_PROPS["target"]] = safe_property_value(types[QUEUE_PROPS["target"]], target_page_ids, QUEUE_PROPS["target"])
    props[QUEUE_PROPS["scheduled_at"]] = safe_property_value(types[QUEUE_PROPS["scheduled_at"]], scheduled_at, QUEUE_PROPS["scheduled_at"])
    props[QUEUE_PROPS["status"]] = safe_property_value(types[QUEUE_PROPS["status"]], status, QUEUE_PROPS["status"])
    props[QUEUE_PROPS["attempts"]] = safe_property_value(types[QUEUE_PROPS["attempts"]], int(attempts), QUEUE_PROPS["attempts"])
    props[QUEUE_PROPS["max_attempts"]] = safe_property_value(types[QUEUE_PROPS["max_attempts"]], int(max_attempts), QUEUE_PROPS["max_attempts"])

    # initialize errors/result fields
    if QUEUE_PROPS["last_error"] in types:
        props[QUEUE_PROPS["last_error"]] = safe_property_value(types[QUEUE_PROPS["last_error"]], "", QUEUE_PROPS["last_error"])
    if QUEUE_PROPS["last_attempt_at"] in types:
        props[QUEUE_PROPS["last_attempt_at"]] = safe_property_value(types[QUEUE_PROPS["last_attempt_at"]], None, QUEUE_PROPS["last_attempt_at"])
    if QUEUE_PROPS["result_id"] in types:
        props[QUEUE_PROPS["result_id"]] = safe_property_value(types[QUEUE_PROPS["result_id"]], "", QUEUE_PROPS["result_id"])
    if QUEUE_PROPS["result_url"] in types:
        props[QUEUE_PROPS["result_url"]] = safe_property_value(types[QUEUE_PROPS["result_url"]], None, QUEUE_PROPS["result_url"])

    # Plan A
    if run_id and QUEUE_PROPS["run_id"] in types:
        props[QUEUE_PROPS["run_id"]] = safe_property_value(types[QUEUE_PROPS["run_id"]], run_id, QUEUE_PROPS["run_id"])
    if dedup_key and QUEUE_PROPS["dedup_key"] in types:
        props[QUEUE_PROPS["dedup_key"]] = safe_property_value(types[QUEUE_PROPS["dedup_key"]], dedup_key, QUEUE_PROPS["dedup_key"])
    if payload is not None and QUEUE_PROPS["payload"] in types:
        payload_str = _json.dumps(payload, ensure_ascii=False)
        # Notion rich_text practical limit is large, but keep it sane
        if len(payload_str) > 5000:
            payload_str = payload_str[:5000] + "…(truncated)"
        props[QUEUE_PROPS["payload"]] = safe_property_value(types[QUEUE_PROPS["payload"]], payload_str, QUEUE_PROPS["payload"])

    page = notion_client.request("POST", "/pages", json={"parent": {"database_id": MONITORING_QUEUE_DB}, "properties": props})
    logger.info("Created queue item: %s", name)
    return extract_queue_item_properties(page)


def query_queue_by_status(status: str, limit: int = 50) -> List[Dict[str, Any]]:
    """
    Retrieve queue items filtered by Status.
    """
    types = introspect_database_properties(MONITORING_QUEUE_DB, use_cache=True)

    payload: Dict[str, Any] = {
        "page_size": min(limit, 100),
        "sorts": [{"property": QUEUE_PROPS["scheduled_at"], "direction": "ascending"}],
    }
    if QUEUE_PROPS["status"] in types:
        payload["filter"] = {"property": QUEUE_PROPS["status"], "select": {"equals": status}}

    res = query_database(MONITORING_QUEUE_DB, payload)
    results = res.get("results", []) or []
    return [extract_queue_item_properties(p) for p in results]


def query_queue_by_dedup_key(dedup_key: str) -> Optional[Dict[str, Any]]:
    """
    Find a queue item by Dedup Key (recommended for idempotency).
    """
    if not dedup_key:
        return None

    payload = {
        "page_size": 1,
        "filter": {"property": QUEUE_PROPS["dedup_key"], "rich_text": {"equals": dedup_key}},
    }
    res = query_database(MONITORING_QUEUE_DB, payload)
    results = res.get("results", []) or []
    if not results:
        return None
    return extract_queue_item_properties(results[0])


def update_queue_item(page_id: str, updates: Dict[str, Any]) -> Dict[str, Any]:
    """
    updates: {NotionPropertyName: python_value}
    Example:
      {"Status": "PROCESSING", "Attempts": 1, "Last Attempt At": "2026-01-25"}
    """
    if not page_id:
        raise ValueError("page_id is required")

    types = introspect_database_properties(MONITORING_QUEUE_DB, use_cache=True)

    notion_props: Dict[str, Any] = {}
    for prop_name, value in updates.items():
        if prop_name not in types:
            logger.warning("Unknown property '%s' (skipping)", prop_name)
            continue
        notion_props[prop_name] = safe_property_value(types[prop_name], value, prop_name)

    page = notion_client.request("PATCH", f"/pages/{page_id}", json={"properties": notion_props})
    logger.info("Updated queue item: %s", page_id)
    return extract_queue_item_properties(page)


def mark_queue_attempt(
    page_id: str,
    attempts: int,
    status: Optional[str] = None,
    last_error: Optional[str] = None,
    result_id: Optional[str] = None,
    result_url: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Convenience helper:
      - set Attempts
      - set Last Attempt At = now
      - optionally set Status / Last Error / Result ID/URL
    """
    now_iso = datetime.now(timezone.utc).date().isoformat()

    updates: Dict[str, Any] = {
        QUEUE_PROPS["attempts"]: int(attempts),
        QUEUE_PROPS["last_attempt_at"]: now_iso,
    }
    if status is not None:
        updates[QUEUE_PROPS["status"]] = status
    if last_error is not None:
        updates[QUEUE_PROPS["last_error"]] = (last_error or "")[:2000]
    if result_id is not None:
        updates[QUEUE_PROPS["result_id"]] = (result_id or "")[:2000]
    if result_url is not None:
        updates[QUEUE_PROPS["result_url"]] = result_url

    return update_queue_item(page_id, updates)


logger.info("Monitoring Queue DB CRUD wrappers initialized (researchOS schema + Plan A)")


2026-01-27 15:52:27 [INFO] Monitoring Queue DB CRUD wrappers initialized (researchOS schema + Plan A)


In [12]:
# ============================================================
# Cell 12 — Deduplication helper functions (researchOS aligned)
# ============================================================
# Overview:
#   Dedup helpers aligned to researchOS DB schemas (Papers / Events / Queue).
#   Primary strategy: deterministic keys (Dedup Key, Source UID, Source URL).
#   Secondary strategy: lightweight fuzzy similarity (normalized token Jaccard).
#
# Notes:
#   - Prefer deterministic keys for idempotency in daily runs.
#   - Similarity is used only as fallback when key is missing.
#   - All queries use existing wrappers:
#       Papers: query_paper_by_dedup_key(), query_papers_by_title() (optional)
#       Events: query_event_by_dedup_key()
#       Queue : query_queue_by_dedup_key()
#

from typing import Dict, Any, Optional, Tuple, List
import re
from urllib.parse import urlparse, urlunparse

# --------------------------
# String / URL normalization
# --------------------------

def normalize_string(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[,\.;:!?\-–—()\[\]{}\"'`]", "", s)
    return s

def simple_similarity_score(a: str, b: str) -> float:
    a = normalize_string(a)
    b = normalize_string(b)
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    sa = set(a.split())
    sb = set(b.split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / max(1, len(sa | sb))

def is_similar(a: str, b: str, threshold: float = 0.85) -> bool:
    return simple_similarity_score(a, b) >= threshold

def normalize_url(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    p = urlparse(url)
    return urlunparse((
        (p.scheme or "https").lower(),
        (p.netloc or "").lower(),
        (p.path or "").rstrip("/"),
        "", "", ""
    ))

def make_dedup_key(namespace: str, *parts: str) -> str:
    """
    Deterministic dedup key builder (human-readable).
    Example:
      make_dedup_key("paper", source_uid)
      make_dedup_key("event", source_url)
    """
    clean = [normalize_string(x) for x in parts if x]
    base = "|".join(clean)
    # keep it short-ish but stable
    return f"{namespace}:{base}"[:400]


# --------------------------
# Papers dedup (researchOS)
# --------------------------

def find_duplicate_paper(
    name: str,
    dedup_key: Optional[str] = None,
    source_uid: Optional[str] = None,
    pdf_link: Optional[str] = None,
    name_threshold: float = 0.90,
) -> Tuple[bool, Optional[str], Optional[str]]:
    """
    Returns: (is_duplicate, page_id, reason)
    Priority:
      1) Dedup Key exact
      2) Source UID exact (by building dedup_key if you standardize)
      3) PDF Link exact-ish (normalize_url; stored as PDF Link in Papers)
      4) Name fuzzy (last resort)
    """
    if dedup_key:
        ex = query_paper_by_dedup_key(dedup_key)
        if ex:
            return True, ex["page_id"], f"dedup_key:{dedup_key}"

    # If you standardize Source UID -> dedup key, this becomes powerful:
    if source_uid:
        dk2 = make_dedup_key("paper", source_uid)
        ex = query_paper_by_dedup_key(dk2)
        if ex:
            return True, ex["page_id"], f"source_uid->dedup_key:{dk2}"

    # PDF link check (if you have a query by url, use it; otherwise do a contains query fallback)
    if pdf_link:
        # Fallback: title contains query is cheaper than scanning all;
        # we do a small name search and compare normalized pdf_link if present.
        norm_pdf = normalize_url(pdf_link)
        candidates = []
        try:
            candidates = query_papers_by_title((name or "")[:60])  # if you kept this function
        except Exception:
            candidates = []
        for c in candidates:
            # try to compare extracted pdf_link if your extractor includes it
            c_pdf = normalize_url(c.get("pdf_link") or "")
            if norm_pdf and c_pdf and norm_pdf == c_pdf:
                return True, c["page_id"], f"pdf_link:{norm_pdf}"

    # Name fuzzy fallback
    if name:
        candidates = []
        try:
            candidates = query_papers_by_title(name)
        except Exception:
            candidates = []
        for c in candidates:
            if is_similar(name, c.get("name") or c.get("title") or "", threshold=name_threshold):
                return True, c["page_id"], f"name_similarity:{c.get('name') or c.get('title')}"
    return False, None, None


# --------------------------
# Events dedup (researchOS)
# --------------------------

def find_duplicate_event(
    name: str,
    dedup_key: Optional[str] = None,
    source_url: Optional[str] = None,
    date: Optional[str] = None,
    name_threshold: float = 0.88,
) -> Tuple[bool, Optional[str], Optional[str]]:
    """
    Priority:
      1) Dedup Key exact
      2) Source URL exact (recommended -> use as dedup key)
      3) (name + date) fuzzy fallback
    """
    if dedup_key:
        ex = query_event_by_dedup_key(dedup_key)
        if ex:
            return True, ex["page_id"], f"dedup_key:{dedup_key}"

    if source_url:
        dk2 = make_dedup_key("event", normalize_url(source_url))
        ex = query_event_by_dedup_key(dk2)
        if ex:
            return True, ex["page_id"], f"source_url->dedup_key:{dk2}"

    # Fallback: name contains + compare
    if name:
        # minimal query: use data_sources query directly (if you want), but reuse get_recent_events is not good for search.
        # Here we do a light search by Name contains.
        payload = {"page_size": 10, "filter": {"property": EVENT_PROPS["name"], "title": {"contains": name[:80]}}}
        res = query_database(EVENTS_DB_ID, payload)
        results = res.get("results", []) or []
        for p in results:
            ex = extract_event_properties(p)
            if not ex.get("name"):
                continue
            if date and ex.get("date") and ex.get("date") != date:
                continue
            if is_similar(name, ex["name"], threshold=name_threshold):
                return True, ex["page_id"], f"name_similarity:{ex['name']}"
    return False, None, None


# --------------------------
# Queue dedup (researchOS)
# --------------------------

def find_duplicate_queue_item(
    name: str,
    dedup_key: Optional[str] = None,
) -> Tuple[bool, Optional[str], Optional[str]]:
    """
    Queue should be strict: prefer Dedup Key only.
    If you don't have it, you risk duplicate enqueues.
    """
    if dedup_key:
        ex = query_queue_by_dedup_key(dedup_key)
        if ex:
            return True, ex["page_id"], f"dedup_key:{dedup_key}"
    return False, None, None


# --------------------------
# Batch utilities
# --------------------------

def batch_deduplicate_papers(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    new_items, dups = [], []
    for it in items:
        is_dup, page_id, reason = find_duplicate_paper(
            name=it.get("name") or it.get("title") or "",
            dedup_key=it.get("dedup_key"),
            source_uid=it.get("source_uid"),
            pdf_link=it.get("pdf_link"),
        )
        if is_dup:
            dups.append({"item": it, "existing_page_id": page_id, "reason": reason})
        else:
            new_items.append(it)
    return {"new_items": new_items, "duplicates": dups}

def batch_deduplicate_events(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    new_items, dups = [], []
    for it in items:
        is_dup, page_id, reason = find_duplicate_event(
            name=it.get("name") or "",
            dedup_key=it.get("dedup_key"),
            source_url=it.get("source_url"),
            date=it.get("date"),
        )
        if is_dup:
            dups.append({"item": it, "existing_page_id": page_id, "reason": reason})
        else:
            new_items.append(it)
    return {"new_items": new_items, "duplicates": dups}

def batch_deduplicate_queue(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    new_items, dups = [], []
    for it in items:
        is_dup, page_id, reason = find_duplicate_queue_item(
            name=it.get("name") or "",
            dedup_key=it.get("dedup_key"),
        )
        if is_dup:
            dups.append({"item": it, "existing_page_id": page_id, "reason": reason})
        else:
            new_items.append(it)
    return {"new_items": new_items, "duplicates": dups}


logger.info("Deduplication helpers initialized (researchOS aligned; Dedup Key first)")


2026-01-27 15:52:31 [INFO] Deduplication helpers initialized (researchOS aligned; Dedup Key first)


In [13]:
# ============================================================
# Cell 13 — Logging utilities with Run ID tracking (stable ISO-Z)
# ============================================================
# Overview:
#   Logging utilities with stable Run ID (UTC ISO8601 with 'Z') for correlation.
#   Provides:
#     - run_id generation / global context
#     - context manager
#     - structured operation logs + error logs
#
# Notes:
#   - Run ID format: YYYY-MM-DDTHH:MM:SS.ffffffZ (UTC)
#   - get_or_set_run_id() ensures run_id never becomes None during a run
#

from typing import Optional, Dict, Any
from datetime import datetime, timezone
import logging

# --- helpers ---

def _iso_z(dt: datetime) -> str:
    """Return UTC ISO8601 with 'Z' suffix and microseconds."""
    dt = dt.astimezone(timezone.utc)
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def truncate(s: str, n: int = 2000) -> str:
    s = str(s or "")
    return s if len(s) <= n else s[:n] + "…(truncated)"

# --- Run ID generation ---

def generate_run_id() -> str:
    """Generate unique run ID based on current UTC timestamp (ISO-Z)."""
    return _iso_z(datetime.now(timezone.utc))

# --- Global run ID context ---
current_run_id: Optional[str] = None

def set_run_id(run_id: Optional[str] = None) -> str:
    """Set global run ID for current execution context."""
    global current_run_id
    current_run_id = run_id or generate_run_id()
    logger.info("Run ID set: %s", current_run_id)
    return current_run_id

def get_run_id() -> Optional[str]:
    return current_run_id

def get_or_set_run_id() -> str:
    """Return current run_id, creating one if missing."""
    global current_run_id
    if not current_run_id:
        current_run_id = generate_run_id()
        logger.info("Run ID auto-created: %s", current_run_id)
    return current_run_id

# --- Context manager for run ID scope ---

class RunContext:
    """Context manager for automatic run ID scope management."""

    def __init__(self, run_id: Optional[str] = None):
        self.run_id = run_id or generate_run_id()
        self.previous_run_id: Optional[str] = None

    def __enter__(self):
        global current_run_id
        self.previous_run_id = current_run_id
        current_run_id = self.run_id
        logger.info("Entering run context: %s", self.run_id)
        return self.run_id

    def __exit__(self, exc_type, exc_val, exc_tb):
        global current_run_id
        logger.info("Exiting run context: %s", self.run_id)
        current_run_id = self.previous_run_id
        return False  # do not suppress exceptions

# --- Structured logging with run ID ---

def log_operation(
    operation: str,
    status: str,
    details: Optional[Dict[str, Any]] = None,
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    rid = run_id or get_or_set_run_id()
    entry = {
        "timestamp": _iso_z(datetime.now(timezone.utc)),
        "run_id": rid,
        "operation": operation,
        "status": status,
        "details": details or {},
    }

    lvl = logging.INFO if status in ("success", "ok", "done") else logging.WARNING
    logger.log(lvl, "[%s] %s (run: %s)", operation, status, rid)
    return entry

def log_error_with_context(
    error: Exception,
    operation: str,
    context: Optional[Dict[str, Any]] = None,
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    rid = run_id or get_or_set_run_id()
    entry = {
        "timestamp": _iso_z(datetime.now(timezone.utc)),
        "run_id": rid,
        "operation": operation,
        "error_type": type(error).__name__,
        "error_message": truncate(str(error), 2000),
        "context": context or {},
    }

    logger.error(
        "[%s] %s: %s (run: %s)",
        operation,
        entry["error_type"],
        entry["error_message"],
        rid,
    )
    return entry

logger.info("Logging utilities initialized (Run ID: ISO-Z)")
logger.info("Functions: generate_run_id(), set_run_id(), get_or_set_run_id(), log_operation(), log_error_with_context()")
logger.info("Context manager: RunContext")


2026-01-27 15:52:33 [INFO] Logging utilities initialized (Run ID: ISO-Z)
2026-01-27 15:52:33 [INFO] Functions: generate_run_id(), set_run_id(), get_or_set_run_id(), log_operation(), log_error_with_context()
2026-01-27 15:52:33 [INFO] Context manager: RunContext


In [14]:
# ============================================================
# Cell 14 — Diagnostic test (data_sources-aware)
# ============================================================
# Overview:
#   Validate auth, DB access, schema compliance, and property introspection.
#   Works with Notion DBs where schema lives under data_sources.
#
# Notes:
#   - Auth / DB access uses /users/me and /databases/{id}
#   - Schema introspection uses /data_sources/{id}
#   - Handles validate_all_databases() returning either:
#       A) Dict[db_name, SchemaValidationResult-like]
#       B) Dict[db_name, {"is_valid":..., "missing":..., "mismatched":...}]
#

import sys

RUN_FULL_TESTS = False  # keep False unless you add smoke-create w/ cleanup

test_results = {
    "auth": False,
    "db_access": {},
    "schema_validation": {},
    "property_introspection": {},
    "overall_status": "PENDING",
}

print("\n" + "=" * 70)
print("NOTION I/O LAYER DIAGNOSTIC TEST")
print("=" * 70)

# --- helpers ---

def _coerce_validation_result(db_name: str, result: Any) -> Dict[str, Any]:
    """
    Normalize validation results into:
      {"is_valid": bool, "missing_count": int, "mismatch_count": int}
    """
    # Case A: SchemaValidationResult-like
    if hasattr(result, "is_valid"):
        summ = result.get_summary() if hasattr(result, "get_summary") else {}
        return {
            "is_valid": bool(getattr(result, "is_valid", False)),
            "missing_count": int(summ.get("missing_count", len(getattr(result, "missing_properties", []) or []))),
            "mismatch_count": int(summ.get("mismatch_count", len(getattr(result, "type_mismatches", {}) or {}))),
        }

    # Case B: dict
    if isinstance(result, dict):
        missing = result.get("missing") or []
        mismatched = result.get("mismatched") or []
        return {
            "is_valid": bool(result.get("is_valid", (len(missing) == 0 and len(mismatched) == 0))),
            "missing_count": int(len(missing)),
            "mismatch_count": int(len(mismatched)),
        }

    # Unknown
    return {"is_valid": False, "missing_count": -1, "mismatch_count": -1}


print("\n[1/4] Testing authentication...")
try:
    auth_valid, auth_error = validate_notion_auth()
    test_results["auth"] = auth_valid
    if auth_valid:
        print("  ✓ Authentication successful")
    else:
        print(f"  ✗ Authentication failed: {auth_error}")
except Exception as e:
    print(f"  ✗ Authentication test error: {e}")
    test_results["auth"] = False


print("\n[2/4] Testing database access...")
for db_name, schema in SCHEMA_REGISTRY.items():
    db_id = schema["database_id"]
    try:
        accessible, error = validate_database_access(db_id, db_name)
        test_results["db_access"][db_name] = accessible
        status = "✓" if accessible else "✗"
        print(f"  {status} {db_name}: {'accessible' if accessible else error}")
    except Exception as e:
        print(f"  ✗ {db_name}: {e}")
        test_results["db_access"][db_name] = False


print("\n[3/4] Validating database schemas...")
try:
    validation_results = validate_all_databases(strict=False)
    for db_name, result in validation_results.items():
        norm = _coerce_validation_result(db_name, result)
        test_results["schema_validation"][db_name] = norm["is_valid"]
        status = "✓" if norm["is_valid"] else "✗"
        print(f"  {status} {db_name}: {norm['missing_count']} missing, {norm['mismatch_count']} mismatches")
except Exception as e:
    print(f"  ✗ Schema validation error: {e}")


print("\n[4/4] Testing property introspection (data_sources)...")
try:
    # IMPORTANT: this must be data_sources-aware
    # use the function that worked for you:
    #   introspect_properties_from_database(database_id)
    introspection_results = {}
    for db_name, schema in SCHEMA_REGISTRY.items():
        db_id = schema["database_id"]
        prop_types = introspect_properties_from_database(db_id)  # returns {name: type}
        introspection_results[db_name] = prop_types

    for db_name, prop_types in introspection_results.items():
        success = len(prop_types) > 0
        test_results["property_introspection"][db_name] = success
        status = "✓" if success else "✗"
        print(f"  {status} {db_name}: {len(prop_types)} properties detected")
except Exception as e:
    print(f"  ✗ Property introspection error: {e}")


all_tests_passed = (
    test_results["auth"]
    and all(test_results["db_access"].values())
    and all(test_results["schema_validation"].values())
    and all(test_results["property_introspection"].values())
)

test_results["overall_status"] = "PASS" if all_tests_passed else "FAIL"

print("\n" + "=" * 70)
print(f"TEST SUMMARY: {test_results['overall_status']}")
print("=" * 70)
print(f"Authentication: {'✓ PASS' if test_results['auth'] else '✗ FAIL'}")
print(f"Database Access: {sum(test_results['db_access'].values())}/{len(test_results['db_access'])} accessible")
print(f"Schema Validation: {sum(test_results['schema_validation'].values())}/{len(test_results['schema_validation'])} valid")
print(f"Property Introspection: {sum(test_results['property_introspection'].values())}/{len(test_results['property_introspection'])} successful")
print("=" * 70 + "\n")

if not all_tests_passed:
    print("⚠ WARNING: Some tests failed. Review errors above before using I/O layer.\n")
else:
    print("✓ All tests passed. Notion I/O layer is ready for use.\n")



NOTION I/O LAYER DIAGNOSTIC TEST

[1/4] Testing authentication...


2026-01-27 15:52:37 [INFO] Authentication validated: bot (ID: 01c14ba4-e8b5-47cb-818c-c3df8b5b79d9)


  ✓ Authentication successful

[2/4] Testing database access...


2026-01-27 15:52:37 [INFO] Database accessible: papers ('Literature Database')


  ✓ papers: accessible


2026-01-27 15:52:38 [INFO] Database accessible: events ('EVENTS_DB')
2026-01-27 15:52:38 [INFO] Database accessible: monitoring_targets ('MONITORING_TARGETS_DB')


  ✓ events: accessible
  ✓ monitoring_targets: accessible


2026-01-27 15:52:38 [INFO] Database accessible: monitoring_queue ('MONITORING_QUEUE_DB')


  ✓ monitoring_queue: accessible

[3/4] Validating database schemas...
  ✗ Schema validation error: name 'validate_all_databases' is not defined

[4/4] Testing property introspection (data_sources)...
  ✗ Property introspection error: name 'introspect_properties_from_database' is not defined

TEST SUMMARY: PASS
Authentication: ✓ PASS
Database Access: 4/4 accessible
Schema Validation: 0/0 valid
Property Introspection: 0/0 successful

✓ All tests passed. Notion I/O layer is ready for use.



In [15]:
# ============================================================
# Cell 15 — Usage examples and integration notes (researchOS aligned)
# ============================================================
# Overview:
#   Minimal, schema-aligned examples showing how other notebooks should call
#   this Notion I/O layer (Papers / Events / Monitoring Targets / Queue).
#
# Notes:
#   - Always wrap a daily run in RunContext() to propagate Run ID everywhere.
#   - Prefer Dedup Key (deterministic) for idempotency.
#   - Query uses data_sources; create/update uses /pages.
#   - Keep "Payload" small (store a compact JSON string).
#

print("\n[Example 1] Minimal import / init pattern:")
print("""
# In another notebook:
from dotenv import load_dotenv
load_dotenv("env.txt")

# Option A: copy Cells 01-15 into your notebook (fastest while iterating)
# Option B: later, extract these cells into a module and import functions.

# ALWAYS:
with RunContext() as run_id:
    ...
""")

# -----------------------------
# Example 2: Papers ingest (INBOX) with dedup_key + run_id
# -----------------------------
print("\n[Example 2] Ingest a new paper into Papers (INBOX) with dedup:")
print("""
with RunContext() as run_id:
    source_uid = "arxiv:1706.03762"  # or openalex:Wxxxx, doi:..., url hash, etc.
    dedup_key  = make_dedup_key("paper", source_uid)

    is_dup, existing_id, reason = find_duplicate_paper(
        name="Attention Is All You Need",
        dedup_key=dedup_key,
        source_uid=source_uid,
        pdf_link="https://arxiv.org/pdf/1706.03762.pdf",
    )

    if is_dup:
        print("Duplicate paper:", reason, existing_id)
    else:
        # create_paper() should be your schema-aligned wrapper
        # (Name / Authors & Year / Tags / PDF Link / Status / Dedup Key / Source UID / Ingested At / Run ID / PDF Status / Slide 1 URL)
        paper = create_paper(
            name="Attention Is All You Need",
            authors_year="Vaswani et al., 2017",
            tags=["NLP", "Transformers"],
            pdf_link="https://arxiv.org/pdf/1706.03762.pdf",
            status="INBOX",
            dedup_key=dedup_key,
            source_uid=source_uid,
            ingested_at="2026-01-25",
            run_id=run_id,
            pdf_status="PENDING",
            slide_1_url=None,
        )
        print("Created paper page:", paper["page_id"])
""")

# -----------------------------
# Example 3: Queue enqueue (idempotent) for monitoring
# -----------------------------
print("\n[Example 3] Enqueue a monitoring job into Queue (idempotent):")
print("""
with RunContext() as run_id:
    target_id = "<MonitoringTargetPageId>"

    # dedup_key should uniquely represent "this target + this item"
    dedup_key = make_dedup_key("queue", target_id, "news", "2026-01-25")

    ex = query_queue_by_dedup_key(dedup_key)
    if ex:
        print("Queue item already exists:", ex["page_id"])
    else:
        item = create_queue_item(
            name="Check latest updates for target",
            queue_type="NEWS",
            target_page_ids=[target_id],
            scheduled_at="2026-01-25",
            status="NEW",
            attempts=0,
            max_attempts=3,
            run_id=run_id,
            dedup_key=dedup_key,
            payload={
                "target_id": target_id,
                "mode": "news",
                "lookback_hours": 24
            },
        )
        print("Enqueued:", item["page_id"])
""")

# -----------------------------
# Example 4: Events ingest (from monitoring) with dedup via source_url
# -----------------------------
print("\n[Example 4] Create an Event from a detected news item (dedup by Source URL):")
print("""
with RunContext() as run_id:
    source_url = "https://example.com/news/item123"
    dedup_key = make_dedup_key("event", normalize_url(source_url))

    ex = query_event_by_dedup_key(dedup_key)
    if ex:
        print("Duplicate event:", ex["page_id"])
    else:
        event = create_event(
            name="Company X announced Series B",
            date="2026-01-25",
            detected_at="2026-01-25",
            target_page_id="<MonitoringTargetPageId>",
            event_type="FUNDING",
            source_url=source_url,
            source="WEB",
            summary="Short summary here",
            confidence=0.8,
            status="INBOX",
            run_id=run_id,
            ingested_at="2026-01-25",
            action_needed=True,
            related_paper_page_ids=[],
            dedup_key=dedup_key,
        )
        print("Created event:", event["page_id"])
""")

# -----------------------------
# Example 5: Error handling pattern
# -----------------------------
print("\n[Example 5] Error handling pattern:")
print("""
with RunContext() as run_id:
    try:
        # do something that calls Notion
        ...
    except ValueError as e:
        # Input/schema/value issue (manual fix)
        log_error_with_context(e, operation="some_operation", context={"run_id": run_id})
    except RuntimeError as e:
        # API issues (might be retryable at lower layer)
        log_error_with_context(e, operation="notion_api", context={"run_id": run_id})
        raise
""")

print("\\n" + "="*70)
print("INTEGRATION CHECKLIST (Daily-ready)")
print("="*70)
print("""
1) Wrap each daily run:
   - with RunContext() as run_id:

2) Always set deterministic keys:
   - Papers : Dedup Key = make_dedup_key("paper", Source UID)
   - Events : Dedup Key = make_dedup_key("event", normalize_url(Source URL))
   - Queue  : Dedup Key = make_dedup_key("queue", Target, ItemType, Day/Window)

3) Before bulk runs:
   - Run Cell 14 diagnostics (auth/access/schema/introspection)

4) Write observability fields (Plan A):
   - Papers: Run ID, Ingested At, PDF Status, Slide 1 URL
   - Events: Run ID, Ingested At, Action Needed, Related Papers
   - Queue : Dedup Key, Payload, Last Attempt At, Result ID/URL

5) Keep Payload small:
   - store a compact JSON (truncate if needed)

6) Prefer schema-aligned wrappers:
   - create_paper/create_event/create_queue_item should match your Notion property names.
""")
print("="*70 + "\\n")

logger.info("Usage examples displayed (researchOS aligned)")


2026-01-27 15:52:53 [INFO] Usage examples displayed (researchOS aligned)



[Example 1] Minimal import / init pattern:

# In another notebook:
from dotenv import load_dotenv
load_dotenv("env.txt")

# Option A: copy Cells 01-15 into your notebook (fastest while iterating)
# Option B: later, extract these cells into a module and import functions.

# ALWAYS:
with RunContext() as run_id:
    ...


[Example 2] Ingest a new paper into Papers (INBOX) with dedup:

with RunContext() as run_id:
    source_uid = "arxiv:1706.03762"  # or openalex:Wxxxx, doi:..., url hash, etc.
    dedup_key  = make_dedup_key("paper", source_uid)

    is_dup, existing_id, reason = find_duplicate_paper(
        name="Attention Is All You Need",
        dedup_key=dedup_key,
        source_uid=source_uid,
        pdf_link="https://arxiv.org/pdf/1706.03762.pdf",
    )

    if is_dup:
        print("Duplicate paper:", reason, existing_id)
    else:
        # create_paper() should be your schema-aligned wrapper
        # (Name / Authors & Year / Tags / PDF Link / Status / Dedup Key / Source UID